<a href="https://colab.research.google.com/github/hamed7pix/Data-Structures-and-Algorithms/blob/main/Data_Structures_Tutorial.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Data Structures from Scratch — A Python Tutorial

This notebook builds the fundamental data structures of an undergraduate **Data Structures** course *from scratch* in Python.
Every structure follows the same pattern:

1. **Idea** — what problem the structure solves, explained with a picture or analogy.
2. **Implementation** — a complete, readable Python class (no library shortcuts for the structure itself).
3. **Operation tour** — a runnable example for each common operation:
   - 🔎 **Access / peek / indexing**
   - 🔍 **Search / lookup / find**
   - ➕ **Insert / prepend / append**
   - ➖ **Delete / remove**
   - 🔁 **Traversal, update, and specialized operations** (e.g. *find-min* in a heap, *union* in disjoint sets)
4. **Complexity table** — **time complexity first**, then space.

### Contents

| Part | Topics |
|---|---|
| **0. Warm-up** | Big-O notation, amortized cost, helper utilities |
| **1. Basic data structures** | 1.1 Stack · 1.2 Queue · 1.3 Linked List |
| **2. Tree data structures** | 2.1 Tree implementations, traversals & structural induction · 2.2 Expression trees & notation conversion · 2.3 Trie · 2.4 Binary Search Tree · 2.5 Priority Queue (Min/Max Heap) |
| **3. Advanced data structures** | 3.1 Disjoint Sets (Union–Find) · 3.2 Red-Black Tree · 3.3 Interval Tree |
| **4. Summary** | Master complexity cheat-sheet and exercises |

> **How to use:** run the cells top to bottom (`Kernel → Restart & Run All`). Later sections reuse helpers defined earlier (e.g. the tree printer), so run in order. Only the Python standard library is used.

---
# Part 0 — Warm-up: How do we measure a data structure?

## 0.1 Time complexity and Big-O

We describe how the running time $T(n)$ of an operation **grows** as the input size $n$ grows, ignoring constant factors.

| Notation | Meaning (informally) | Example |
|---|---|---|
| $O(f(n))$ | grows **at most** like $f(n)$ (upper bound) | linear search is $O(n)$ |
| $\Omega(f(n))$ | grows **at least** like $f(n)$ (lower bound) | comparison sorting is $\Omega(n \log n)$ |
| $\Theta(f(n))$ | grows **exactly** like $f(n)$ (tight bound) | summing a list is $\Theta(n)$ |

Formally, $T(n) = O(f(n))$ if there exist constants $c > 0$ and $n_0$ such that $T(n) \le c\,f(n)$ for all $n \ge n_0$.

### Common growth rates (from fastest to slowest)

| Class | Name | Typical operation in this notebook |
|---|---|---|
| $O(1)$ | constant | stack push/pop, heap peek, array indexing |
| $O(\alpha(n))$ | inverse Ackermann ($\le 4$ in practice) | union–find with both optimizations |
| $O(\log n)$ | logarithmic | balanced BST search, heap insert |
| $O(L)$ | length of the key | trie insert/search (independent of how many words are stored!) |
| $O(n)$ | linear | linked-list search, tree traversal |
| $O(n \log n)$ | linearithmic | heapsort, building a BST from random keys |
| $O(n^2)$ | quadratic | inserting $n$ items at the front of a Python list |

### Worst, average and amortized cost
- **Worst case** — the maximum over all inputs of size $n$ (e.g. BST search on a degenerate tree is $O(n)$).
- **Average case** — expected cost over random inputs (BST search on random keys is $O(\log n)$).
- **Amortized** — average cost per operation over a *sequence* of operations. A dynamic array occasionally doubles its capacity in $O(n)$, but $n$ appends cost $O(n)$ in total, so each append is **$O(1)$ amortized**.



In [4]:
# ---- Shared helpers used throughout the notebook ----------------------------
import random, timeit, re, math
from collections import deque, defaultdict

random.seed(42)

def banner(title):
    """Print a section header inside the operation tours."""
    print(f"\n▶ {title}")

def draw_binary(root, label=lambda n: str(n.key), left="left", right="right",
                is_empty=None, tags=("L", "R")):
    """Print any binary tree sideways-top-down with ASCII connectors.

    Works for every node class in this notebook: pass the attribute names of the
    children and, for sentinel-based trees, a custom `is_empty` test.
    """
    is_empty = is_empty or (lambda n: n is None)
    if is_empty(root):
        print("(empty tree)")
        return
    lines = [label(root)]
    def walk(node, prefix):
        kids = [(tag, child) for tag, child in zip(tags, (getattr(node, left), getattr(node, right)))
                if not is_empty(child)]
        for i, (tag, child) in enumerate(kids):
            last = (i == len(kids) - 1)
            lines.append(prefix + ("└── " if last else "├── ") + f"{tag}: " + label(child))
            walk(child, prefix + ("    " if last else "│   "))
    walk(root, "")
    print("\n".join(lines))

def time_op(stmt, number=1, **names):
    """Return the best-of-3 time in milliseconds for running `stmt` `number` times."""
    return min(timeit.repeat(stmt, number=number, repeat=3, globals=names)) * 1000

print("helpers ready")

helpers ready


---
# Part 1 — Basic Data Structures

## 1.1 Stack (LIFO — Last In, First Out)

Think of a stack of plates: you can only add (**push**) or remove (**pop**) the plate on **top**.

```
push(10) push(20) push(30)        pop() → 30
   ┌────┐                            ┌────┐
   │ 30 │ ← top                      │ 20 │ ← top
   │ 20 │                            │ 10 │
   │ 10 │                            └────┘
   └────┘
```

**Where stacks appear:** function call stacks, undo/redo, bracket matching, depth-first search, expression evaluation (Section 2.2).

We implement the stack on a Python list, using the **end** of the list as the top, because `append`/`pop` at the end are $O(1)$. We also give a linked-list-based version, which needs no resizing at all.

In [8]:
class Stack:
    """LIFO stack backed by a dynamic array. The END of the list is the top."""

    def __init__(self, items=None):
        self._data = list(items) if items else []

    # ---- insert ----
    def push(self, item):                 # O(1) amortized
        self._data.append(item)

    # ---- delete ----
    def pop(self):                        # O(1)
        if self.is_empty():
            raise IndexError("pop from empty stack")
        return self._data.pop()

    # ---- access ----
    def peek(self):                       # O(1): look at the top without removing it
        if self.is_empty():
            raise IndexError("peek from empty stack")
        return self._data[-1]


    def is_empty(self):
        return not self._data

    def __len__(self):
        return len(self._data)



class LinkedStack:
    """LIFO stack built from linked nodes: the head node is the top. No resizing ever."""

    class _Node:
        __slots__ = ("value", "next")
        def __init__(self, value, next=None):
            self.value, self.next = value, next

    def __init__(self):
        self._top, self._size = None, 0

    def push(self, item):                 # O(1) worst case
        self._top = self._Node(item, self._top)
        self._size += 1

    def pop(self):                        # O(1) worst case
        if self._top is None:
            raise IndexError("pop from empty stack")
        value, self._top = self._top.value, self._top.next
        self._size -= 1
        return value

    def peek(self):                       # O(1)
        if self._top is None:
            raise IndexError("peek from empty stack")
        return self._top.value

    def __len__(self):
        return self._size


In [9]:
s = Stack()

banner("Insert: push 10, 20, 30, 40")
for x in (10, 20, 30, 40):
    s.push(x)
print(s)

banner("Access: peek()")
print("peek()      =", s.peek())


banner("Delete: pop() twice")
print("popped", s.pop(), "then", s.pop(), "->", s)


banner("Update (replace the top): pop then push")
s.pop(); s.push(25)
print(s)

banner("Same operations on LinkedStack")
ls = LinkedStack()
for x in "ABC":
    ls.push(x)
print(ls)
print("peek =", ls.peek())
print("pop  =", ls.pop(), "| now:", ls)

banner("Error handling: pop from an empty stack")
try:
    Stack().pop()
except IndexError as e:
    print("IndexError:", e)


▶ Insert: push 10, 20, 30, 40

▶ Access: peek()
peek()      = 40

▶ Delete: pop() twice
popped 40 then 30 -> <__main__.Stack object at 0x7fc60d8f6900>

▶ Update (replace the top): pop then push

▶ Same operations on LinkedStack
peek = C
pop  = C | now: <__main__.LinkedStack object at 0x7fc60d8f6a50>

▶ Error handling: pop from an empty stack
IndexError: pop from empty stack


### Applications of a stack

**1. Balanced brackets.** Scan left to right; push every opening bracket; on a closing bracket the top of the stack must be its partner. One pass → $O(n)$ time, $O(n)$ space.

**2. Reversing a sequence.** Push everything, pop everything.

In [10]:
def is_balanced(text):
    pairs = {")": "(", "]": "[", "}": "{"}
    st = Stack()
    for ch in text:
        if ch in "([{":
            st.push(ch)
        elif ch in pairs:
            if st.is_empty() or st.pop() != pairs[ch]:
                return False
    return st.is_empty()

for expr in ["(a+b)*[c-d]", "{[()()]}", "(]", "((x)", "f(a[i]) + {g}"]:
    print(f"{expr:<16} balanced? {is_balanced(expr)}")

def reverse_with_stack(seq):
    st = Stack(seq)
    return "".join(st.pop() for _ in range(len(st)))

print("\nreverse('stack') =", reverse_with_stack("stack"))

(a+b)*[c-d]      balanced? True
{[()()]}         balanced? True
(]               balanced? False
((x)             balanced? False
f(a[i]) + {g}    balanced? True

reverse('stack') = kcats


### Stack — complexity

| Operation | Array-based `Stack` | `LinkedStack` |
|---|---|---|
| `push` (insert) | $O(1)$ amortized ($O(n)$ on a resize) | $O(1)$ worst case |
| `pop` (delete) | $O(1)$ | $O(1)$ |
| `peek` (access top) | $O(1)$ | $O(1)$ |
| `peek_at(k)` (access depth $k$) | $O(1)$ | $O(k)$ |
| `search` | $O(n)$ | $O(n)$ |
| traversal | $O(n)$ | $O(n)$ |
| **Space** | $O(n)$ (plus unused capacity) | $O(n)$ (plus one pointer per item) |

## 1.2 Queue (FIFO — First In, First Out)

A line at a ticket office: new people join at the **rear** (**enqueue**), service happens at the **front** (**dequeue**).

```
enqueue →  [ rear ... 30  20  10  front ]  → dequeue (returns 10)
```

**Where queues appear:** task scheduling, printer jobs, buffering, breadth-first search (used later for level-order traversal).

### Why not just use a Python list?
`list.pop(0)` removes the first element and **shifts all remaining elements**: $O(n)$ per dequeue. Let's measure it against `collections.deque`, which is designed for $O(1)$ operations at both ends.

In [ ]:
print(f"{'n':>8} | {'list.pop(0) (ms)':>17} | {'deque.popleft() (ms)':>20}")
for n in [10_000, 20_000, 40_000, 80_000]:
    t_list  = time_op("a=list(range(n))\nwhile a: a.pop(0)", n=n)
    t_deque = time_op("d=deque(range(n))\nwhile d: d.popleft()", n=n, deque=deque)
    print(f"{n:>8} | {t_list:>17.2f} | {t_deque:>20.2f}")

       n |  list.pop(0) (ms) | deque.popleft() (ms)
   10000 |              6.09 |                 0.35
   20000 |             34.40 |                 0.96


   40000 |            121.09 |                 1.27


   80000 |            507.94 |                 3.95


### Implementation 1: circular array (ring buffer)

We keep a fixed-size array plus two pieces of state: the index of the **front** and the current **size**. The rear position is computed as `(front + size) % capacity`, so the queue "wraps around" the end of the array. When the array is full we double it — just like a dynamic array — giving $O(1)$ amortized enqueue.

```
capacity 8, front = 6, size = 4        logical order: A B C D
index:   0   1   2   3   4   5   6   7
       [ C | D | . | . | . | . | A | B ]
                                ^front
```

In [ ]:
class CircularQueue:
    """FIFO queue on a circular array that doubles when full."""

    def __init__(self, capacity=4):
        self._data = [None] * capacity
        self._front = 0
        self._size = 0

    # ---- insert ----
    def enqueue(self, item):                       # O(1) amortized
        if self._size == len(self._data):
            self._resize(2 * len(self._data))
        rear = (self._front + self._size) % len(self._data)
        self._data[rear] = item
        self._size += 1

    # ---- delete ----
    def dequeue(self):                             # O(1)
        if self.is_empty():
            raise IndexError("dequeue from empty queue")
        item = self._data[self._front]
        self._data[self._front] = None             # help the garbage collector
        self._front = (self._front + 1) % len(self._data)
        self._size -= 1
        return item

    # ---- access ----
    def peek(self):                                # O(1): the front item
        if self.is_empty():
            raise IndexError("peek from empty queue")
        return self._data[self._front]

    def peek_rear(self):                           # O(1): the most recently added item
        if self.is_empty():
            raise IndexError("peek from empty queue")
        return self._data[(self._front + self._size - 1) % len(self._data)]

    def __getitem__(self, i):                      # O(1): i-th item counted from the front
        if not 0 <= i < self._size:
            raise IndexError("queue index out of range")
        return self._data[(self._front + i) % len(self._data)]

    # ---- search ----
    def search(self, item):                        # O(n): position from the front, -1 if absent
        for i in range(self._size):
            if self[i] == item:
                return i
        return -1

    def _resize(self, new_capacity):               # O(n), happens rarely
        old = [self[i] for i in range(self._size)]
        self._data = old + [None] * (new_capacity - self._size)
        self._front = 0

    def is_empty(self):
        return self._size == 0

    def __len__(self):
        return self._size

    def __iter__(self):                            # traversal front → rear
        for i in range(self._size):
            yield self[i]

    def __repr__(self):
        return f"Queue(front→rear: {list(self)})"

    def debug(self):
        return f"raw={self._data}  front={self._front}  size={self._size}"

In [ ]:
q = CircularQueue(capacity=4)

banner("Insert: enqueue A, B, C")
for x in "ABC":
    q.enqueue(x)
print(q, "|", q.debug())

banner("Delete: dequeue twice (front moves forward)")
print("dequeued", q.dequeue(), q.dequeue(), "|", q.debug())

banner("Insert: enqueue D, E, F (wraps around the end of the array)")
for x in "DEF":
    q.enqueue(x)
print(q, "|", q.debug())

banner("Insert: enqueue G, H (array full -> doubles to capacity 8)")
q.enqueue("G"); q.enqueue("H")
print(q, "|", q.debug())

banner("Access: peek(), peek_rear(), q[2]")
print("front =", q.peek(), "| rear =", q.peek_rear(), "| q[2] =", q[2])

banner("Search: position of 'F' and 'Z'")
print("search('F') =", q.search("F"), "| search('Z') =", q.search("Z"))

banner("Traversal: front → rear")
print(" ".join(q))


▶ Insert: enqueue A, B, C
Queue(front→rear: ['A', 'B', 'C']) | raw=['A', 'B', 'C', None]  front=0  size=3

▶ Delete: dequeue twice (front moves forward)
dequeued A B | raw=[None, None, 'C', None]  front=2  size=1

▶ Insert: enqueue D, E, F (wraps around the end of the array)
Queue(front→rear: ['C', 'D', 'E', 'F']) | raw=['E', 'F', 'C', 'D']  front=2  size=4

▶ Insert: enqueue G, H (array full -> doubles to capacity 8)
Queue(front→rear: ['C', 'D', 'E', 'F', 'G', 'H']) | raw=['C', 'D', 'E', 'F', 'G', 'H', None, None]  front=0  size=6

▶ Access: peek(), peek_rear(), q[2]
front = C | rear = H | q[2] = E

▶ Search: position of 'F' and 'Z'
search('F') = 3 | search('Z') = -1

▶ Traversal: front → rear
C D E F G H


### Implementation 2: queue from two stacks (amortized analysis)

Keep an **in-stack** for enqueues and an **out-stack** for dequeues. When the out-stack is empty, pour the whole in-stack into it (reversing the order, so the oldest element ends on top).

Each element is moved **at most once** from `in` to `out`, so $n$ operations cost $O(n)$ in total → **$O(1)$ amortized** per operation, even though a single dequeue can cost $O(n)$.

In [ ]:
class TwoStackQueue:
    def __init__(self):
        self._in, self._out = [], []

    def enqueue(self, x):            # O(1)
        self._in.append(x)

    def _shift(self):                # move elements only when out-stack is empty
        if not self._out:
            while self._in:
                self._out.append(self._in.pop())

    def dequeue(self):               # O(1) amortized
        self._shift()
        if not self._out:
            raise IndexError("dequeue from empty queue")
        return self._out.pop()

    def peek(self):                  # O(1) amortized
        self._shift()
        if not self._out:
            raise IndexError("peek from empty queue")
        return self._out[-1]

    def __len__(self):
        return len(self._in) + len(self._out)

tq = TwoStackQueue()
for x in range(1, 6):
    tq.enqueue(x)
print("peek =", tq.peek(), "| internal in/out:", tq._in, tq._out)
print("dequeue ->", tq.dequeue(), tq.dequeue())
tq.enqueue(6)
print("after enqueue(6): in =", tq._in, " out =", tq._out)
print("drain:", [tq.dequeue() for _ in range(len(tq))])

banner("In practice: Python's collections.deque")
d = deque([1, 2, 3])
d.append(4); d.appendleft(0)        # insert at both ends, O(1)
print(d, "| popleft =", d.popleft(), "| pop =", d.pop(), "| d[1] =", d[1])

peek = 1 | internal in/out: [] [5, 4, 3, 2, 1]
dequeue -> 1 2
after enqueue(6): in = [6]  out = [5, 4, 3]
drain: [3, 4, 5, 6]

▶ In practice: Python's collections.deque
deque([1, 2, 3]) | popleft = 0 | pop = 4 | d[1] = 2


### Application: Round-robin CPU scheduling

Each job runs for at most a *quantum* of time, then goes to the back of the queue if unfinished.

In [ ]:
def round_robin(jobs, quantum):
    q = CircularQueue()
    for name, need in jobs:
        q.enqueue((name, need))
    clock, finished = 0, []
    while q:
        name, remaining = q.dequeue()
        run = min(quantum, remaining)
        clock += run
        remaining -= run
        if remaining:
            q.enqueue((name, remaining))
        else:
            finished.append((name, clock))
    return finished

for name, t in round_robin([("P1", 5), ("P2", 3), ("P3", 8), ("P4", 2)], quantum=2):
    print(f"{name} finished at t={t}")

P4 finished at t=8
P2 finished at t=11
P1 finished at t=14
P3 finished at t=18


### Queue — complexity

| Operation | `list` + `pop(0)` | `CircularQueue` | `TwoStackQueue` | `deque` |
|---|---|---|---|---|
| enqueue (insert) | $O(1)$ amortized | $O(1)$ amortized | $O(1)$ | $O(1)$ |
| dequeue (delete) | **$O(n)$** | $O(1)$ | $O(1)$ amortized | $O(1)$ |
| peek (access front) | $O(1)$ | $O(1)$ | $O(1)$ amortized | $O(1)$ |
| index $i$ | $O(1)$ | $O(1)$ | $O(n)$ | $O(n)$ (fast near ends) |
| search | $O(n)$ | $O(n)$ | $O(n)$ | $O(n)$ |
| **Space** | $O(n)$ | $O(n)$ | $O(n)$ | $O(n)$ |

## 1.3 Linked List

An array stores elements in **contiguous** memory, so element $i$ is found by arithmetic in $O(1)$. A linked list stores each element in a separate **node** that points to the next node:

```
head                                   tail
 │                                       │
 ▼                                       ▼
[ 3 | •]──► [ 7 | •]──► [ 1 | •]──► [ 9 | None]
```

Trade-off:
- ✅ inserting/removing at a **known position** (e.g. the head) is $O(1)$ — no shifting.
- ❌ reaching the $i$-th element requires **walking** $i$ links: $O(n)$.

We implement a **singly linked list** with both `head` and `tail` pointers (so `append` is $O(1)$), then a **doubly linked list** with a sentinel node.

In [ ]:
class ListNode:
    __slots__ = ("value", "next")
    def __init__(self, value, next=None):
        self.value, self.next = value, next


class SinglyLinkedList:
    def __init__(self, iterable=()):
        self.head = None
        self.tail = None
        self._size = 0
        for x in iterable:
            self.append(x)

    # ------------------------------------------------------------ helpers
    def __len__(self):
        return self._size

    def __iter__(self):                          # traversal, O(n)
        node = self.head
        while node:
            yield node.value
            node = node.next

    def __repr__(self):
        return " -> ".join(map(str, self)) + " -> None" if self.head else "(empty)"

    def _node_at(self, index):                   # walk `index` links: O(index)
        if not 0 <= index < self._size:
            raise IndexError("list index out of range")
        node = self.head
        for _ in range(index):
            node = node.next
        return node

    # ------------------------------------------------------------ access
    def get(self, index):                        # O(n)
        return self._node_at(index).value

    __getitem__ = get

    def first(self):                             # O(1)
        if not self.head: raise IndexError("empty list")
        return self.head.value

    def last(self):                              # O(1) thanks to the tail pointer
        if not self.tail: raise IndexError("empty list")
        return self.tail.value

    # ------------------------------------------------------------ update
    def set(self, index, value):                 # O(n)
        self._node_at(index).value = value

    # ------------------------------------------------------------ search
    def find(self, value):                       # O(n): index of first match, -1 if absent
        for i, v in enumerate(self):
            if v == value:
                return i
        return -1

    def __contains__(self, value):
        return self.find(value) != -1

    # ------------------------------------------------------------ insert
    def prepend(self, value):                    # O(1)
        self.head = ListNode(value, self.head)
        if self.tail is None:
            self.tail = self.head
        self._size += 1

    def append(self, value):                     # O(1)
        node = ListNode(value)
        if self.tail is None:
            self.head = self.tail = node
        else:
            self.tail.next = node
            self.tail = node
        self._size += 1

    def insert(self, index, value):              # O(index)
        if not 0 <= index <= self._size:
            raise IndexError("insert index out of range")
        if index == 0:
            return self.prepend(value)
        if index == self._size:
            return self.append(value)
        prev = self._node_at(index - 1)
        prev.next = ListNode(value, prev.next)
        self._size += 1

    # ------------------------------------------------------------ delete
    def pop_front(self):                         # O(1)
        if self.head is None:
            raise IndexError("pop from empty list")
        value = self.head.value
        self.head = self.head.next
        if self.head is None:
            self.tail = None
        self._size -= 1
        return value

    def pop_back(self):                          # O(n): must find the node BEFORE the tail
        if self.head is None:
            raise IndexError("pop from empty list")
        if self._size == 1:
            return self.pop_front()
        prev = self._node_at(self._size - 2)
        value = self.tail.value
        prev.next = None
        self.tail = prev
        self._size -= 1
        return value

    def delete_at(self, index):                  # O(index)
        if not 0 <= index < self._size:
            raise IndexError("list index out of range")
        if index == 0:
            return self.pop_front()
        prev = self._node_at(index - 1)
        target = prev.next
        prev.next = target.next
        if target is self.tail:
            self.tail = prev
        self._size -= 1
        return target.value

    def remove(self, value):                     # O(n): delete first occurrence
        prev, node = None, self.head
        while node:
            if node.value == value:
                if prev is None:
                    self.head = node.next
                else:
                    prev.next = node.next
                if node is self.tail:
                    self.tail = prev
                self._size -= 1
                return True
            prev, node = node, node.next
        return False

    # ------------------------------------------------------------ specialized
    def reverse(self):                           # O(n) time, O(1) extra space
        prev, node = None, self.head
        self.tail = self.head
        while node:
            nxt = node.next
            node.next = prev
            prev, node = node, nxt
        self.head = prev

In [ ]:
ll = SinglyLinkedList([3, 7, 1])
print("start:", ll, "| size =", len(ll))

banner("Insert: prepend(0), append(9), insert(2, 5)")
ll.prepend(0); print("prepend(0)   ->", ll)
ll.append(9);  print("append(9)    ->", ll)
ll.insert(2, 5); print("insert(2, 5) ->", ll)

banner("Access / indexing: ll[0], ll[3], first(), last()")
print(ll[0], ll[3], ll.first(), ll.last())

banner("Search: find(1), find(42), 7 in ll")
print(ll.find(1), ll.find(42), 7 in ll)

banner("Update: set(1, 33)")
ll.set(1, 33); print(ll)

banner("Delete: pop_front(), pop_back(), delete_at(1), remove(1)")
print("pop_front() =", ll.pop_front(), "| list now:", ll)
print("pop_back()  =", ll.pop_back(), "| list now:", ll)
print("delete_at(1)=", ll.delete_at(1), "| list now:", ll)
print("remove(1)   =", ll.remove(1), "| list now:", ll, "| tail =", ll.last())

banner("Traversal + specialized: reverse in place")
ll2 = SinglyLinkedList(range(1, 7))
print("before:", ll2)
ll2.reverse()
print("after :", ll2, "| head =", ll2.first(), "tail =", ll2.last())

start: 3 -> 7 -> 1 -> None | size = 3

▶ Insert: prepend(0), append(9), insert(2, 5)
prepend(0)   -> 0 -> 3 -> 7 -> 1 -> None
append(9)    -> 0 -> 3 -> 7 -> 1 -> 9 -> None
insert(2, 5) -> 0 -> 3 -> 5 -> 7 -> 1 -> 9 -> None

▶ Access / indexing: ll[0], ll[3], first(), last()
0 7 0 9

▶ Search: find(1), find(42), 7 in ll
4 -1 True

▶ Update: set(1, 33)
0 -> 33 -> 5 -> 7 -> 1 -> 9 -> None

▶ Delete: pop_front(), pop_back(), delete_at(1), remove(1)
pop_front() = 0 | list now: 33 -> 5 -> 7 -> 1 -> 9 -> None
pop_back()  = 9 | list now: 33 -> 5 -> 7 -> 1 -> None
delete_at(1)= 5 | list now: 33 -> 7 -> 1 -> None
remove(1)   = True | list now: 33 -> 7 -> None | tail = 7

▶ Traversal + specialized: reverse in place
before: 1 -> 2 -> 3 -> 4 -> 5 -> 6 -> None
after : 6 -> 5 -> 4 -> 3 -> 2 -> 1 -> None | head = 6 tail = 1


### Doubly linked list with a sentinel

Each node also points to its **previous** node. A single dummy **sentinel** node sits between the tail and the head, making the list circular:

```
       ┌────────────────────────────────────────────┐
       ▼                                            │
 [sentinel] ⇄ [ A ] ⇄ [ B ] ⇄ [ C ] ⇄ (back to sentinel)
```

Benefits:
- No special cases for "empty list" or "first/last node" — there is always a node before and after.
- Removing a node **you already hold a reference to** is $O(1)$ (needed by LRU caches, for example).
- `pop_back` becomes $O(1)$.
- Indexing walks from the nearer end: $O(\min(i, n-i))$.

In [ ]:
class DNode:
    __slots__ = ("value", "prev", "next")
    def __init__(self, value, prev=None, next=None):
        self.value, self.prev, self.next = value, prev, next


class DoublyLinkedList:
    def __init__(self, iterable=()):
        self._s = DNode(None)            # sentinel
        self._s.prev = self._s.next = self._s
        self._size = 0
        for x in iterable:
            self.append(x)

    def __len__(self):
        return self._size

    def __iter__(self):
        node = self._s.next
        while node is not self._s:
            yield node.value
            node = node.next

    def __reversed__(self):
        node = self._s.prev
        while node is not self._s:
            yield node.value
            node = node.prev

    def __repr__(self):
        return "None <- " + " <-> ".join(map(str, self)) + " -> None"

    # ---- core O(1) primitives ----
    def _insert_between(self, value, before, after):
        node = DNode(value, before, after)
        before.next = node
        after.prev = node
        self._size += 1
        return node

    def remove_node(self, node):          # O(1) given the node
        node.prev.next = node.next
        node.next.prev = node.prev
        self._size -= 1
        return node.value

    # ---- insert ----
    def append(self, value):              # O(1)
        return self._insert_between(value, self._s.prev, self._s)

    def prepend(self, value):             # O(1)
        return self._insert_between(value, self._s, self._s.next)

    def insert(self, index, value):       # O(min(i, n-i))
        if index == self._size:
            return self.append(value)
        after = self._node_at(index)
        return self._insert_between(value, after.prev, after)

    # ---- access ----
    def _node_at(self, index):
        if not 0 <= index < self._size:
            raise IndexError("index out of range")
        if index < self._size // 2:       # walk forward from the head
            node = self._s.next
            for _ in range(index):
                node = node.next
        else:                             # walk backward from the tail
            node = self._s.prev
            for _ in range(self._size - 1 - index):
                node = node.prev
        return node

    def __getitem__(self, index):
        return self._node_at(index).value

    # ---- search ----
    def find_node(self, value):           # O(n)
        node = self._s.next
        while node is not self._s:
            if node.value == value:
                return node
            node = node.next
        return None

    # ---- delete ----
    def pop_front(self):                  # O(1)
        if not self._size: raise IndexError("empty")
        return self.remove_node(self._s.next)

    def pop_back(self):                   # O(1)  (O(n) in a singly linked list!)
        if not self._size: raise IndexError("empty")
        return self.remove_node(self._s.prev)

    def remove(self, value):              # O(n) search + O(1) unlink
        node = self.find_node(value)
        if node is None:
            return False
        self.remove_node(node)
        return True


dl = DoublyLinkedList("BCD")
banner("Insert: prepend('A'), append('E'), insert(2, 'X')")
dl.prepend("A"); dl.append("E"); nx = dl.insert(2, "X")
print(dl)

banner("Access: dl[0], dl[4] (walks from the tail)")
print(dl[0], dl[4])

banner("Search: find_node('D')")
node = dl.find_node("D")
print("found node with value", node.value, "| prev =", node.prev.value, "| next =", node.next.value)

banner("Delete: remove_node(<X node>) in O(1), pop_front(), pop_back(), remove('C')")
dl.remove_node(nx); print("remove X   ->", dl)
dl.pop_front();     print("pop_front  ->", dl)
dl.pop_back();      print("pop_back   ->", dl)
dl.remove("C");     print("remove('C')->", dl)

banner("Traversal in both directions")
dl2 = DoublyLinkedList(range(5))
print("forward :", list(dl2), "\nbackward:", list(reversed(dl2)))


▶ Insert: prepend('A'), append('E'), insert(2, 'X')
None <- A <-> B <-> X <-> C <-> D <-> E -> None

▶ Access: dl[0], dl[4] (walks from the tail)
A D

▶ Search: find_node('D')
found node with value D | prev = C | next = E

▶ Delete: remove_node(<X node>) in O(1), pop_front(), pop_back(), remove('C')
remove X   -> None <- A <-> B <-> C <-> D <-> E -> None
pop_front  -> None <- B <-> C <-> D <-> E -> None
pop_back   -> None <- B <-> C <-> D -> None
remove('C')-> None <- B <-> D -> None

▶ Traversal in both directions
forward : [0, 1, 2, 3, 4] 
backward: [4, 3, 2, 1, 0]


### Specialized: cycle detection (Floyd's tortoise and hare)

A bug or a malicious input can produce a list whose last node points back into the list. Two pointers — one moving 1 step, one moving 2 steps — meet **iff** there is a cycle. $O(n)$ time, $O(1)$ space.

In [ ]:
def has_cycle(head):
    slow = fast = head
    while fast and fast.next:
        slow, fast = slow.next, fast.next.next
        if slow is fast:
            return True
    return False

nodes = [ListNode(i) for i in range(6)]
for a, b in zip(nodes, nodes[1:]):
    a.next = b
print("straight list has cycle?", has_cycle(nodes[0]))
nodes[-1].next = nodes[2]          # 5 -> 2 creates a loop
print("after linking 5 -> 2   ?", has_cycle(nodes[0]))

straight list has cycle? False
after linking 5 -> 2   ? True


### Linked list vs dynamic array — complexity

| Operation | Dynamic array (`list`) | Singly linked (head+tail) | Doubly linked (sentinel) |
|---|---|---|---|
| access index $i$ | **$O(1)$** | $O(i)$ | $O(\min(i, n-i))$ |
| search by value | $O(n)$ | $O(n)$ | $O(n)$ |
| insert at front (prepend) | $O(n)$ | **$O(1)$** | **$O(1)$** |
| insert at end (append) | $O(1)$ amortized | $O(1)$ | $O(1)$ |
| insert at index $i$ | $O(n - i)$ | $O(i)$ | $O(\min(i, n-i))$ |
| delete front | $O(n)$ | $O(1)$ | $O(1)$ |
| delete end | $O(1)$ | **$O(n)$** | $O(1)$ |
| delete a node you hold | — | $O(n)$ (need predecessor) | **$O(1)$** |
| **Space** | $O(n)$, compact | $O(n)$ + 1 pointer/node | $O(n)$ + 2 pointers/node |

**Rule of thumb:** arrays win for random access and cache-friendliness; linked lists win when you insert/delete at positions you already hold.

---
# Part 2 — Tree Data Structures

## 2.1 Trees: vocabulary, implementations, traversals and structural induction

A **tree** is a hierarchy: one **root**, and every other node has exactly one **parent**.

```
                 A            ← root (depth 0)
             /   |   \
            B    C    D       ← children of A (depth 1)
          /  \        |
         E    F       G       ← leaves E, F, G (no children)
```

| Term | Meaning |
|---|---|
| **parent / child** | B is the parent of E; E is a child of B |
| **siblings** | nodes with the same parent (B, C, D) |
| **leaf** | node with no children; **internal node** has ≥ 1 child |
| **depth** of a node | number of edges from the root to it |
| **height** of a node | number of edges on the longest path down to a leaf (leaf = 0; empty tree = −1) |
| **subtree** | a node together with all of its descendants |
| **degree** | number of children of a node |
| **binary tree** | every node has at most two children, called **left** and **right** |

We look at four ways to *store* a tree:

1. **General tree** — each node keeps a list of children.
2. **Left-child / right-sibling (LCRS)** — any tree stored with just two pointers per node.
3. **Array-based binary tree** — children of index $i$ live at $2i+1$ and $2i+2$ (used by heaps).
4. **Linked binary tree** — each node has `left` and `right` pointers (used by BSTs, expression trees, red-black trees).

### Implementation 1: General tree (list of children)

Example: a tiny file system. Operations:
- **access**: the root, a child by position, a node's parent — $O(1)$
- **search**: depth-first search for a value — $O(n)$
- **insert**: `add_child` — $O(1)$
- **delete**: remove a whole subtree from its parent — $O(\text{degree})$
- **traversal**: pre-order, post-order, level-order — $O(n)$

In [ ]:
class TreeNode:
    def __init__(self, value):
        self.value = value
        self.children = []
        self.parent = None

    def add_child(self, value):                 # insert, O(1)
        child = TreeNode(value)
        child.parent = self
        self.children.append(child)
        return child

    def __repr__(self):
        return f"TreeNode({self.value!r})"


def gt_find(node, target):                      # search, O(n) depth-first
    if node.value == target:
        return node
    for child in node.children:
        found = gt_find(child, target)
        if found:
            return found
    return None

def gt_remove_subtree(node):                    # delete, O(degree of the parent)
    if node.parent:
        node.parent.children.remove(node)
        node.parent = None

def gt_size(node):
    return 1 + sum(gt_size(c) for c in node.children)

def gt_height(node):                            # leaf has height 0
    return 0 if not node.children else 1 + max(gt_height(c) for c in node.children)

def gt_depth(node):                             # walk up through parents, O(depth)
    d = 0
    while node.parent:
        node, d = node.parent, d + 1
    return d

def gt_path(node):                              # the path from the root to the node
    parts = []
    while node:
        parts.append(node.value)
        node = node.parent
    return "/".join(reversed(parts))

def gt_preorder(node):                          # parent BEFORE children
    yield node.value
    for c in node.children:
        yield from gt_preorder(c)

def gt_postorder(node):                         # parent AFTER children
    for c in node.children:
        yield from gt_postorder(c)
    yield node.value

def gt_levelorder(node):                        # breadth-first, uses a queue
    q = deque([node])
    while q:
        cur = q.popleft()
        yield cur.value
        q.extend(cur.children)

def draw_general(node, prefix="", last=True, is_root=True):
    print(("" if is_root else prefix + ("└── " if last else "├── ")) + str(node.value))
    new_prefix = "" if is_root else prefix + ("    " if last else "│   ")
    for i, c in enumerate(node.children):
        draw_general(c, new_prefix, i == len(node.children) - 1, False)


root = TreeNode("root")
home = root.add_child("home")
etc = root.add_child("etc")
alice = home.add_child("alice")
alice.add_child("notes.txt"); alice.add_child("photo.png")
home.add_child("bob").add_child("todo.md")
etc.add_child("hosts")
draw_general(root)

banner("Access: root.children[0], a node's parent")
print(root.children[0], "| parent of 'alice' =", alice.parent)

banner("Search: gt_find('todo.md') and its path / depth")
n = gt_find(root, "todo.md")
print(n, "| path =", gt_path(n), "| depth =", gt_depth(n))

banner("Size and height")
print("size =", gt_size(root), "| height =", gt_height(root))

banner("Traversals")
print("pre-order  :", list(gt_preorder(root)))
print("post-order :", list(gt_postorder(root)))
print("level-order:", list(gt_levelorder(root)))

banner("Insert: add 'passwd' under etc;  Delete: remove the 'home/bob' subtree")
etc.add_child("passwd")
gt_remove_subtree(gt_find(root, "bob"))
draw_general(root)

root
├── home
│   ├── alice
│   │   ├── notes.txt
│   │   └── photo.png
│   └── bob
│       └── todo.md
└── etc
    └── hosts

▶ Access: root.children[0], a node's parent
TreeNode('home') | parent of 'alice' = TreeNode('home')

▶ Search: gt_find('todo.md') and its path / depth
TreeNode('todo.md') | path = root/home/bob/todo.md | depth = 3

▶ Size and height
size = 9 | height = 3

▶ Traversals
pre-order  : ['root', 'home', 'alice', 'notes.txt', 'photo.png', 'bob', 'todo.md', 'etc', 'hosts']
post-order : ['notes.txt', 'photo.png', 'alice', 'todo.md', 'bob', 'home', 'hosts', 'etc', 'root']
level-order: ['root', 'home', 'etc', 'alice', 'bob', 'hosts', 'notes.txt', 'photo.png', 'todo.md']

▶ Insert: add 'passwd' under etc;  Delete: remove the 'home/bob' subtree
root
├── home
│   └── alice
│       ├── notes.txt
│       └── photo.png
└── etc
    ├── hosts
    └── passwd


### Implementation 2: Left-child / right-sibling (any tree → binary tree)

A node with many children needs a variable-length list. LCRS uses **two fixed pointers**:
- `first_child` → the leftmost child
- `next_sibling` → the next node to the right with the same parent

So every tree can be stored as a *binary* tree. To list the children of a node, follow `first_child` once, then `next_sibling` repeatedly.

In [ ]:
class LCRSNode:
    def __init__(self, value):
        self.value = value
        self.first_child = None
        self.next_sibling = None

def to_lcrs(gnode):
    """Convert a general TreeNode into LCRS form. O(n)."""
    node = LCRSNode(gnode.value)
    prev = None
    for child in gnode.children:
        c = to_lcrs(child)
        if prev is None:
            node.first_child = c
        else:
            prev.next_sibling = c
        prev = c
    return node

def lcrs_children(node):
    c = node.first_child
    while c:
        yield c.value
        c = c.next_sibling

lroot = to_lcrs(root)
print("Children of root via LCRS:", list(lcrs_children(lroot)))
print("\nThe same tree drawn as a binary tree (C = first child, S = next sibling):")
draw_binary(lroot, label=lambda n: str(n.value), left="first_child", right="next_sibling", tags=("C", "S"))

Children of root via LCRS: ['home', 'etc']

The same tree drawn as a binary tree (C = first child, S = next sibling):
root
└── C: home
    ├── C: alice
    │   └── C: notes.txt
    │       └── S: photo.png
    └── S: etc
        └── C: hosts
            └── S: passwd


### Implementation 3: Array-based binary tree

Number the nodes level by level, left to right, starting at 0. Then for node $i$:

$$\text{left}(i) = 2i+1,\qquad \text{right}(i) = 2i+2,\qquad \text{parent}(i) = \left\lfloor \tfrac{i-1}{2} \right\rfloor$$

```
index:      0   1   2   3   4   5   6
array:    [ A,  B,  C,  D,  E, None, F ]

                 A(0)
               /      \
           B(1)        C(2)
          /   \           \
       D(3)   E(4)        F(6)
```

- ✅ No pointers; navigation is pure arithmetic; **$O(1)$ access by position**.
- ✅ Perfect for **complete** trees (every level full except the last, filled from the left) — this is exactly how heaps are stored (Section 2.5).
- ❌ Wastes space for sparse trees: a degenerate chain of height $h$ needs $2^{h+1}-1$ slots.

In [ ]:
class ArrayBinaryTree:
    def __init__(self, values=()):
        self.a = list(values)                 # None marks a missing node

    @staticmethod
    def left(i):   return 2 * i + 1
    @staticmethod
    def right(i):  return 2 * i + 2
    @staticmethod
    def parent(i): return (i - 1) // 2 if i > 0 else None

    def exists(self, i):
        return 0 <= i < len(self.a) and self.a[i] is not None

    def get(self, i):                         # access, O(1)
        if not self.exists(i):
            raise IndexError(f"no node at index {i}")
        return self.a[i]

    def set(self, i, value):                  # update, O(1)
        if not self.exists(i):
            raise IndexError(f"no node at index {i}")
        self.a[i] = value

    def find(self, value):                    # search, O(n)
        for i, v in enumerate(self.a):
            if v == value:
                return i
        return -1

    def insert_level_order(self, value):      # insert into the first free slot, O(n)
        for i, v in enumerate(self.a):
            if v is None and (i == 0 or self.exists(self.parent(i))):
                self.a[i] = value
                return i
        self.a.append(value)                  # tree is complete: O(1) amortized
        return len(self.a) - 1

    def delete_subtree(self, i):              # delete node i and all descendants, O(size of subtree)
        if not self.exists(i):
            return
        stack = [i]
        while stack:
            j = stack.pop()
            if j < len(self.a) and self.a[j] is not None:
                self.a[j] = None
                stack.extend((self.left(j), self.right(j)))
        while self.a and self.a[-1] is None:  # trim trailing holes
            self.a.pop()

    def to_linked(self, i=0):                 # convert to linked BinaryNodes (defined below)
        if not self.exists(i):
            return None
        return BinaryNode(self.a[i], self.to_linked(self.left(i)), self.to_linked(self.right(i)))

### Implementation 4: Linked binary tree + the four classic traversals

| Traversal | Order | Typical use |
|---|---|---|
| **Pre-order** | node, left, right | copy a tree, prefix notation |
| **In-order** | left, node, right | sorted output of a BST, infix notation |
| **Post-order** | left, right, node | delete/free a tree, evaluate expressions, postfix notation |
| **Level-order** | level by level (BFS) | shortest path in unweighted trees, printing by level |

All four visit every node once: **$O(n)$ time**. Recursive versions use $O(h)$ stack space ($h$ = height); level-order uses $O(w)$ queue space ($w$ = maximum width). We show both recursive and iterative (explicit stack) versions — the iterative ones are a nice application of Section 1.1.

In [ ]:
class BinaryNode:
    def __init__(self, key, left=None, right=None):
        self.key, self.left, self.right = key, left, right
    def __repr__(self):
        return f"BinaryNode({self.key!r})"

# ---------- recursive traversals ----------
def preorder(n):
    if n:
        yield n.key
        yield from preorder(n.left)
        yield from preorder(n.right)

def inorder(n):
    if n:
        yield from inorder(n.left)
        yield n.key
        yield from inorder(n.right)

def postorder(n):
    if n:
        yield from postorder(n.left)
        yield from postorder(n.right)
        yield n.key

def level_order(n):
    """Return a list of levels, each a list of keys."""
    if not n:
        return []
    levels, q = [], deque([n])
    while q:
        level = []
        for _ in range(len(q)):
            cur = q.popleft()
            level.append(cur.key)
            if cur.left:  q.append(cur.left)
            if cur.right: q.append(cur.right)
        levels.append(level)
    return levels

# ---------- iterative traversals (explicit stack) ----------
def preorder_iter(root):
    out, st = [], [root] if root else []
    while st:
        n = st.pop()
        out.append(n.key)
        if n.right: st.append(n.right)    # push right first so left is processed first
        if n.left:  st.append(n.left)
    return out

def inorder_iter(root):
    out, st, n = [], [], root
    while st or n:
        while n:                          # go as far left as possible
            st.append(n)
            n = n.left
        n = st.pop()
        out.append(n.key)
        n = n.right
    return out

def postorder_iter(root):                 # two-stack method
    out, st = [], [root] if root else []
    while st:
        n = st.pop()
        out.append(n.key)                 # produces node, right, left ...
        if n.left:  st.append(n.left)
        if n.right: st.append(n.right)
    return out[::-1]                      # ... reversed = left, right, node

# ---------- other basic operations ----------
def tree_size(n):
    return 0 if n is None else 1 + tree_size(n.left) + tree_size(n.right)

def tree_height(n):                       # empty tree = -1, single node = 0
    return -1 if n is None else 1 + max(tree_height(n.left), tree_height(n.right))

def count_leaves(n):
    if n is None: return 0
    if not n.left and not n.right: return 1
    return count_leaves(n.left) + count_leaves(n.right)

def bt_find(n, key):                      # search in an UNORDERED binary tree: O(n)
    if n is None or n.key == key:
        return n
    return bt_find(n.left, key) or bt_find(n.right, key)

def bt_insert_level_order(root, key):     # insert into the first free spot (keeps it complete)
    node = BinaryNode(key)
    if root is None:
        return node
    q = deque([root])
    while q:
        cur = q.popleft()
        if cur.left is None:
            cur.left = node; return root
        if cur.right is None:
            cur.right = node; return root
        q.extend((cur.left, cur.right))

def bt_delete(root, key):
    """Delete `key` from an unordered binary tree: overwrite it with the deepest,
    right-most node's key and remove that node. O(n)."""
    if root is None:
        return None
    if root.left is None and root.right is None:
        return None if root.key == key else root
    target, last, parent_of_last = None, None, None
    q = deque([(root, None)])
    while q:
        node, parent = q.popleft()
        if node.key == key and target is None:
            target = node
        last, parent_of_last = node, parent
        if node.left:  q.append((node.left, node))
        if node.right: q.append((node.right, node))
    if target is None:
        return root
    target.key = last.key
    if parent_of_last.left is last:
        parent_of_last.left = None
    else:
        parent_of_last.right = None
    return root

In [ ]:
abt = ArrayBinaryTree(["A", "B", "C", "D", "E", None, "F"])

banner("Array tree — access by index arithmetic")
i = abt.find("E")
print(f"'E' is at index {i}; parent index {abt.parent(i)} -> {abt.get(abt.parent(i))}; "
      f"children of 'C' (index 2): left exists? {abt.exists(abt.left(2))}, right = {abt.get(abt.right(2))}")

banner("Array tree — insert_level_order('G') fills the hole at index 5")
print("inserted at index", abt.insert_level_order("G"), "->", abt.a)

banner("Array tree -> linked tree, then draw it")
t = abt.to_linked()
draw_binary(t)

banner("Traversals (recursive vs iterative give the same answers)")
print("pre-order  :", list(preorder(t)),  preorder_iter(t))
print("in-order   :", list(inorder(t)),   inorder_iter(t))
print("post-order :", list(postorder(t)), postorder_iter(t))
print("level-order:", level_order(t))

banner("Size / height / leaves / search")
print("size =", tree_size(t), "| height =", tree_height(t), "| leaves =", count_leaves(t),
      "| find('F') =", bt_find(t, "F"), "| find('Z') =", bt_find(t, "Z"))

banner("Insert 'H' (level order), then delete 'B' (replaced by deepest node 'H')")
t = bt_insert_level_order(t, "H")
print("after insert:", level_order(t))
t = bt_delete(t, "B")
print("after delete:", level_order(t))
draw_binary(t)

banner("Array tree — delete_subtree at index 1 (B's whole subtree)")
abt.delete_subtree(1)
print(abt.a)


▶ Array tree — access by index arithmetic
'E' is at index 4; parent index 1 -> B; children of 'C' (index 2): left exists? False, right = F

▶ Array tree — insert_level_order('G') fills the hole at index 5
inserted at index 5 -> ['A', 'B', 'C', 'D', 'E', 'G', 'F']

▶ Array tree -> linked tree, then draw it
A
├── L: B
│   ├── L: D
│   └── R: E
└── R: C
    ├── L: G
    └── R: F

▶ Traversals (recursive vs iterative give the same answers)
pre-order  : ['A', 'B', 'D', 'E', 'C', 'G', 'F'] ['A', 'B', 'D', 'E', 'C', 'G', 'F']
in-order   : ['D', 'B', 'E', 'A', 'G', 'C', 'F'] ['D', 'B', 'E', 'A', 'G', 'C', 'F']
post-order : ['D', 'E', 'B', 'G', 'F', 'C', 'A'] ['D', 'E', 'B', 'G', 'F', 'C', 'A']
level-order: [['A'], ['B', 'C'], ['D', 'E', 'G', 'F']]

▶ Size / height / leaves / search
size = 7 | height = 2 | leaves = 4 | find('F') = BinaryNode('F') | find('Z') = None

▶ Insert 'H' (level order), then delete 'B' (replaced by deepest node 'H')
after insert: [['A'], ['B', 'C'], ['D', 'E', 'G', 'F']

### Specialized: rebuilding a tree from two traversals

A single traversal does not identify a binary tree, but **pre-order + in-order** (with distinct keys) does:
the first pre-order element is the root; its position in the in-order list splits the left and right subtrees. Using a dictionary of in-order positions gives $O(n)$ time.

In [ ]:
def build_from_pre_in(pre, ino):
    pos = {k: i for i, k in enumerate(ino)}
    it = iter(pre)
    def build(lo, hi):                    # build the subtree whose in-order slice is ino[lo:hi]
        if lo >= hi:
            return None
        key = next(it)
        mid = pos[key]
        node = BinaryNode(key)
        node.left = build(lo, mid)
        node.right = build(mid + 1, hi)
        return node
    return build(0, len(ino))

rebuilt = build_from_pre_in(list("ABDECF"), list("DBEACF"))
draw_binary(rebuilt)
print("post-order of rebuilt tree:", "".join(postorder(rebuilt)))

A
├── L: B
│   ├── L: D
│   └── R: E
└── R: C
    └── R: F
post-order of rebuilt tree: DEBFCA


### Structural induction

Trees are **defined recursively**:

> A **binary tree** is either (base) the **empty tree** $\varnothing$, or (step) a **node** $r$ with a left binary tree $T_L$ and a right binary tree $T_R$.

**Structural induction** proves a property $P(T)$ for *all* binary trees by following this definition:

1. **Base case:** prove $P(\varnothing)$.
2. **Inductive step:** assuming $P(T_L)$ and $P(T_R)$ (the **inductive hypothesis**), prove $P$ for the tree with root $r$ and subtrees $T_L, T_R$.

It is the tree-shaped version of ordinary induction on $n$ — and it is exactly how we argue that a **recursive function on trees is correct** (the function's recursive calls *are* the inductive hypothesis).

---

**Theorem 1 (null links).** A binary tree with $n$ nodes has exactly $n+1$ empty subtrees (`None` pointers).

*Base:* $\varnothing$ has $n = 0$ nodes and is itself $1 = 0 + 1$ empty subtree. ✓
*Step:* For $T = (r, T_L, T_R)$ with $n = 1 + n_L + n_R$: by hypothesis $T_L$ has $n_L+1$ and $T_R$ has $n_R+1$ empty subtrees, so $T$ has $(n_L+1)+(n_R+1) = n + 1$. ✓

---

**Theorem 2 (size vs height).** A binary tree of height $h$ (empty tree: $h=-1$) has at most $2^{h+1}-1$ nodes.

*Base:* $\varnothing$: $h=-1$, $n = 0 \le 2^0 - 1 = 0$. ✓
*Step:* Both subtrees have height $\le h-1$, so by hypothesis each has $\le 2^{h}-1$ nodes; thus $n \le 1 + 2(2^h - 1) = 2^{h+1}-1$. ✓

*Consequence:* $h \ge \log_2(n+1) - 1$, i.e. **every binary tree with $n$ nodes has height $\Omega(\log n)$** — the best a search tree can hope for.

---

**Theorem 3 (full binary trees).** In a *full* binary tree (every node has 0 or 2 children), $\#\text{leaves} = \#\text{internal} + 1$.

*Base:* a single node: 1 leaf, 0 internal nodes. ✓
*Step:* root with two full subtrees: $L = L_1 + L_2 = (I_1+1) + (I_2+1) = (I_1+I_2+1)+1 = I + 1$, since the root is internal. ✓
(Expression trees with binary operators are full: operands = operators + 1.)

---

**Theorem 4 (correctness of `tree_size`).** `tree_size(T)` returns the number of nodes of $T$.
*Base:* returns 0 for `None`. *Step:* by hypothesis the recursive calls return $n_L$ and $n_R$, so it returns $1 + n_L + n_R = n$. ✓

Let's check all four theorems empirically on thousands of random trees.

In [ ]:
def random_binary_tree(n):
    """Random shape with exactly n nodes."""
    if n == 0:
        return None
    left_size = random.randint(0, n - 1)
    return BinaryNode(n, random_binary_tree(left_size), random_binary_tree(n - 1 - left_size))

def random_full_tree(internal):
    """Random full binary tree with the given number of internal nodes."""
    if internal == 0:
        return BinaryNode("leaf")
    k = random.randint(0, internal - 1)
    return BinaryNode("op", random_full_tree(k), random_full_tree(internal - 1 - k))

def count_null_links(n):
    return 1 if n is None else count_null_links(n.left) + count_null_links(n.right)

def count_nodes_iter(root):            # independent (iterative) count to check Theorem 4
    c, st = 0, [root] if root else []
    while st:
        x = st.pop(); c += 1
        st.extend(ch for ch in (x.left, x.right) if ch)
    return c

ok = True
for _ in range(3000):
    n = random.randint(0, 60)
    T = random_binary_tree(n)
    ok &= count_null_links(T) == n + 1                      # Theorem 1
    ok &= n <= 2 ** (tree_height(T) + 1) - 1                # Theorem 2
    ok &= tree_size(T) == count_nodes_iter(T) == n          # Theorem 4
    F = random_full_tree(random.randint(0, 30))
    ok &= count_leaves(F) == tree_size(F) - count_leaves(F) + 1   # Theorem 3
print("All four theorems hold on 3000 random trees:", ok)

All four theorems hold on 3000 random trees: True


### General & binary tree — complexity

| Operation | General tree (child list) | Array binary tree | Linked binary tree (unordered) |
|---|---|---|---|
| access root / child / parent | $O(1)$ | $O(1)$ (index arithmetic) | $O(1)$ child; parent needs a parent pointer |
| search by value | $O(n)$ | $O(n)$ | $O(n)$ |
| insert child / level-order insert | $O(1)$ | $O(1)$ amortized if complete | $O(n)$ (BFS for the free spot) |
| delete | $O(\text{degree})$ to detach a subtree | $O(\text{subtree size})$ | $O(n)$ (BFS for the deepest node) |
| traversal (any order) | $O(n)$ | $O(n)$ | $O(n)$ |
| **Space** | $O(n)$ | $O(2^{h})$ — $O(n)$ only if complete | $O(n)$ |

## 2.2 Expression trees and converting between notations

The arithmetic expression $(3 + x) \times (8 - 2)$ can be drawn as a binary tree: **operators are internal nodes, operands are leaves**.

```
            *
          /   \
         +     -
        / \   / \
       3   x 8   2
```

The three traversals of this one tree give the three classic notations:

| Notation | Rule | Traversal | Example |
|---|---|---|---|
| **Infix** | operator *between* operands (needs parentheses) | in-order | `(3 + x) * (8 - 2)` |
| **Prefix** (Polish) | operator *before* operands | pre-order | `* + 3 x - 8 2` |
| **Postfix** (Reverse Polish) | operator *after* operands | post-order | `3 x + 8 2 - *` |

Prefix and postfix **never need parentheses**: the order of operators fully determines the structure. That's why compilers and calculators convert infix to postfix and evaluate with a stack.

### Plan
1. **Tokenize** a string into numbers, variables, operators and parentheses.
2. **Infix → postfix** with Dijkstra's **shunting-yard** algorithm (handles precedence and associativity).
3. **Build a tree** from postfix (stack of subtrees) or from prefix (scan right to left).
4. Produce **any notation** with a traversal, and **evaluate** with a post-order traversal.
5. Direct **string-to-string conversions** with a stack (no tree needed).

Operators supported: `+ -` (precedence 1), `* /` (precedence 2), `^` (precedence 3, **right-associative**: `2^3^2 = 2^(3^2)`).

In [ ]:
TOKEN_RE = re.compile(r"\s*(\d+\.\d+|\d+|[A-Za-z_]\w*|[-+*/^()])")
PREC = {"+": 1, "-": 1, "*": 2, "/": 2, "^": 3}
RIGHT_ASSOC = {"^"}

def tokenize(expr):
    """Split an expression into tokens. Tokens in prefix/postfix input must be space-separated."""
    tokens, pos, expr = [], 0, expr.strip()
    while pos < len(expr):
        m = TOKEN_RE.match(expr, pos)
        if not m:
            raise ValueError(f"unexpected character {expr[pos]!r} at position {pos}")
        tokens.append(m.group(1))
        pos = m.end()
    return tokens

def is_operator(tok):
    return tok in PREC

def infix_to_postfix(expr):
    """Shunting-yard algorithm. O(n) where n = number of tokens."""
    out, ops = [], []                                   # ops is a stack of operators
    for tok in tokenize(expr):
        if is_operator(tok):
            # pop operators that must be applied BEFORE tok
            while ops and ops[-1] != "(" and (
                PREC[ops[-1]] > PREC[tok] or
                (PREC[ops[-1]] == PREC[tok] and tok not in RIGHT_ASSOC)):
                out.append(ops.pop())
            ops.append(tok)
        elif tok == "(":
            ops.append(tok)
        elif tok == ")":
            while ops and ops[-1] != "(":
                out.append(ops.pop())
            if not ops:
                raise ValueError("mismatched ')'")
            ops.pop()                                   # discard '('
        else:
            out.append(tok)                             # operand goes straight to output
    while ops:
        op = ops.pop()
        if op == "(":
            raise ValueError("mismatched '('")
        out.append(op)
    return out

def shunting_yard_trace(expr):
    """Same algorithm, printing the stack after each token (for learning)."""
    out, ops = [], []
    print(f"{'token':>6} | {'operator stack':<14} | output")
    for tok in tokenize(expr):
        if is_operator(tok):
            while ops and ops[-1] != "(" and (PREC[ops[-1]] > PREC[tok] or
                    (PREC[ops[-1]] == PREC[tok] and tok not in RIGHT_ASSOC)):
                out.append(ops.pop())
            ops.append(tok)
        elif tok == "(":
            ops.append(tok)
        elif tok == ")":
            while ops[-1] != "(":
                out.append(ops.pop())
            ops.pop()
        else:
            out.append(tok)
        print(f"{tok:>6} | {' '.join(ops):<14} | {' '.join(out)}")
    while ops:
        out.append(ops.pop())
    print(f"{'end':>6} | {'':<14} | {' '.join(out)}")

shunting_yard_trace("3 + 4 * 2 / (1 - 5) ^ 2 ^ 3")

 token | operator stack | output
     3 |                | 3
     + | +              | 3
     4 | +              | 3 4
     * | + *            | 3 4
     2 | + *            | 3 4 2
     / | + /            | 3 4 2 *
     ( | + / (          | 3 4 2 *
     1 | + / (          | 3 4 2 * 1
     - | + / ( -        | 3 4 2 * 1
     5 | + / ( -        | 3 4 2 * 1 5
     ) | + /            | 3 4 2 * 1 5 -
     ^ | + / ^          | 3 4 2 * 1 5 -
     2 | + / ^          | 3 4 2 * 1 5 - 2
     ^ | + / ^ ^        | 3 4 2 * 1 5 - 2
     3 | + / ^ ^        | 3 4 2 * 1 5 - 2 3
   end |                | 3 4 2 * 1 5 - 2 3 ^ ^ / +


In [ ]:
class ExprNode:
    def __init__(self, value, left=None, right=None):
        self.value, self.left, self.right = value, left, right
    def is_leaf(self):
        return self.left is None and self.right is None
    def __repr__(self):
        return f"ExprNode({self.value!r})"

# ---------------------------------------------------------------- build
def build_from_postfix(postfix):
    """Scan left→right; operands are pushed, an operator pops its two operands. O(n)."""
    tokens = tokenize(postfix) if isinstance(postfix, str) else list(postfix)
    st = []
    for tok in tokens:
        if is_operator(tok):
            if len(st) < 2:
                raise ValueError(f"not enough operands for {tok!r}")
            right, left = st.pop(), st.pop()            # note the order!
            st.append(ExprNode(tok, left, right))
        else:
            st.append(ExprNode(tok))
    if len(st) != 1:
        raise ValueError("malformed postfix expression")
    return st[0]

def build_from_prefix(prefix):
    """Scan right→left; same idea, but the first pop is the LEFT operand. O(n)."""
    tokens = tokenize(prefix) if isinstance(prefix, str) else list(prefix)
    st = []
    for tok in reversed(tokens):
        if is_operator(tok):
            left, right = st.pop(), st.pop()
            st.append(ExprNode(tok, left, right))
        else:
            st.append(ExprNode(tok))
    if len(st) != 1:
        raise ValueError("malformed prefix expression")
    return st[0]

def build_from_infix(infix):
    return build_from_postfix(infix_to_postfix(infix))

# ---------------------------------------------------------------- traversals = notations
def to_prefix(n):                                       # pre-order
    return n.value if n.is_leaf() else f"{n.value} {to_prefix(n.left)} {to_prefix(n.right)}"

def to_postfix(n):                                      # post-order
    return n.value if n.is_leaf() else f"{to_postfix(n.left)} {to_postfix(n.right)} {n.value}"

def to_infix(n, minimal=True):
    """In-order. minimal=True adds only the parentheses that precedence/associativity require."""
    if n.is_leaf():
        return n.value
    left, right = to_infix(n.left, minimal), to_infix(n.right, minimal)
    if not minimal:
        return f"({left} {n.value} {right})"
    p = PREC[n.value]
    if not n.left.is_leaf():
        lp = PREC[n.left.value]
        if lp < p or (lp == p and n.value in RIGHT_ASSOC):
            left = f"({left})"
    if not n.right.is_leaf():
        rp = PREC[n.right.value]
        if rp < p or (rp == p and n.value not in RIGHT_ASSOC):
            right = f"({right})"
    return f"{left} {n.value} {right}"

# ---------------------------------------------------------------- evaluate (post-order)
OPS = {"+": lambda a, b: a + b, "-": lambda a, b: a - b,
       "*": lambda a, b: a * b, "/": lambda a, b: a / b, "^": lambda a, b: a ** b}

def evaluate(n, env=None):
    env = env or {}
    if n.is_leaf():
        if n.value in env:
            return env[n.value]
        try:
            return float(n.value)
        except ValueError:
            raise NameError(f"variable {n.value!r} has no value") from None
    return OPS[n.value](evaluate(n.left, env), evaluate(n.right, env))

In [ ]:
expr = "(3 + x) * (8 - 2) / y ^ 2 ^ 1"
tree = build_from_infix(expr)

banner(f"Build from infix:  {expr}")
draw_binary(tree, label=lambda n: n.value)

banner("Traversal → every notation")
print("infix (minimal)   :", to_infix(tree))
print("infix (fully par.):", to_infix(tree, minimal=False))
print("prefix            :", to_prefix(tree))
print("postfix           :", to_postfix(tree))

banner("Round trips: prefix → tree → postfix, and postfix → tree → infix")
t2 = build_from_prefix(to_prefix(tree))
t3 = build_from_postfix(to_postfix(tree))
print(to_postfix(t2) == to_postfix(tree), to_infix(t3) == to_infix(tree))

banner("Evaluate with x = 2, y = 3   (Python check on the right)")
print(evaluate(tree, {"x": 2, "y": 3}), "==", (3 + 2) * (8 - 2) / 3 ** 2 ** 1)

banner("Associativity matters")
for e in ["8 - 4 - 2", "8 - (4 - 2)", "2 ^ 3 ^ 2", "(2 ^ 3) ^ 2"]:
    t = build_from_infix(e)
    print(f"{e:<12} -> postfix {to_postfix(t):<12} -> infix {to_infix(t):<12} = {evaluate(t):g}")


▶ Build from infix:  (3 + x) * (8 - 2) / y ^ 2 ^ 1
/
├── L: *
│   ├── L: +
│   │   ├── L: 3
│   │   └── R: x
│   └── R: -
│       ├── L: 8
│       └── R: 2
└── R: ^
    ├── L: y
    └── R: ^
        ├── L: 2
        └── R: 1

▶ Traversal → every notation
infix (minimal)   : (3 + x) * (8 - 2) / y ^ 2 ^ 1
infix (fully par.): (((3 + x) * (8 - 2)) / (y ^ (2 ^ 1)))
prefix            : / * + 3 x - 8 2 ^ y ^ 2 1
postfix           : 3 x + 8 2 - * y 2 1 ^ ^ /

▶ Round trips: prefix → tree → postfix, and postfix → tree → infix
True True

▶ Evaluate with x = 2, y = 3   (Python check on the right)
3.3333333333333335 == 3.3333333333333335

▶ Associativity matters
8 - 4 - 2    -> postfix 8 4 - 2 -    -> infix 8 - 4 - 2    = 2
8 - (4 - 2)  -> postfix 8 4 2 - -    -> infix 8 - (4 - 2)  = 6
2 ^ 3 ^ 2    -> postfix 2 3 2 ^ ^    -> infix 2 ^ 3 ^ 2    = 512
(2 ^ 3) ^ 2  -> postfix 2 3 ^ 2 ^    -> infix (2 ^ 3) ^ 2  = 64


### Direct conversions with a stack (no explicit tree)

The tree-building algorithms above can push **strings** instead of nodes. This converts between notations in one pass, $O(n)$ stack operations (string concatenation adds cost proportional to the output length).

In [ ]:
def postfix_to_prefix(s):
    st = []
    for tok in tokenize(s):
        if is_operator(tok):
            b, a = st.pop(), st.pop()
            st.append(f"{tok} {a} {b}")
        else:
            st.append(tok)
    return st[0]

def prefix_to_postfix(s):
    st = []
    for tok in reversed(tokenize(s)):
        if is_operator(tok):
            a, b = st.pop(), st.pop()
            st.append(f"{a} {b} {tok}")
        else:
            st.append(tok)
    return st[0]

def postfix_to_infix(s):
    st = []
    for tok in tokenize(s):
        if is_operator(tok):
            b, a = st.pop(), st.pop()
            st.append(f"({a} {tok} {b})")
        else:
            st.append(tok)
    return st[0]

def prefix_to_infix(s):
    st = []
    for tok in reversed(tokenize(s)):
        if is_operator(tok):
            a, b = st.pop(), st.pop()
            st.append(f"({a} {tok} {b})")
        else:
            st.append(tok)
    return st[0]

def infix_to_prefix(s):
    return postfix_to_prefix(" ".join(infix_to_postfix(s)))

def evaluate_postfix(s, env=None):
    """Classic stack evaluator — what a calculator does."""
    env, st = env or {}, []
    for tok in tokenize(s):
        if is_operator(tok):
            b, a = st.pop(), st.pop()
            st.append(OPS[tok](a, b))
        else:
            st.append(env[tok] if tok in env else float(tok))
    return st[0]

src = "a * (b + c) - d / e"
post = " ".join(infix_to_postfix(src))
pre = infix_to_prefix(src)
print("infix            :", src)
print("infix  → postfix :", post)
print("infix  → prefix  :", pre)
print("postfix→ prefix  :", postfix_to_prefix(post))
print("prefix → postfix :", prefix_to_postfix(pre))
print("postfix→ infix   :", postfix_to_infix(post))
print("prefix → infix   :", prefix_to_infix(pre))
print("evaluate postfix with a=2,b=3,c=4,d=10,e=5:",
      evaluate_postfix(post, dict(a=2, b=3, c=4, d=10, e=5)))

infix            : a * (b + c) - d / e
infix  → postfix : a b c + * d e / -
infix  → prefix  : - * a + b c / d e
postfix→ prefix  : - * a + b c / d e
prefix → postfix : a b c + * d e / -
postfix→ infix   : ((a * (b + c)) - (d / e))
prefix → infix   : ((a * (b + c)) - (d / e))
evaluate postfix with a=2,b=3,c=4,d=10,e=5: 12.0


### Operation tour on expression trees

- **Access:** the root is the *last* operator applied; `root.left` / `root.right` are its operands.
- **Search:** find all variables, or test whether a token occurs.
- **Insert / update:** *substitute* a variable with another expression (grafting a subtree).
- **Delete:** *simplify* — constant folding and algebraic identities remove nodes (`x*1 → x`, `x+0 → x`, `x*0 → 0`, `2*3 → 6`).
- **Specialized:** height = how deeply nested the expression is; count of operators.

In [ ]:
def find_variables(n):                                  # search, O(n)
    if n.is_leaf():
        return set() if _is_number(n.value) else {n.value}
    return find_variables(n.left) | find_variables(n.right)

def contains(n, token):                                 # search, O(n)
    return n is not None and (n.value == token or contains(n.left, token) or contains(n.right, token))

def substitute(n, var, replacement):                    # insert/update: returns a NEW tree
    if n.is_leaf():
        return copy_tree(replacement) if n.value == var else ExprNode(n.value)
    return ExprNode(n.value, substitute(n.left, var, replacement), substitute(n.right, var, replacement))

def copy_tree(n):
    return None if n is None else ExprNode(n.value, copy_tree(n.left), copy_tree(n.right))

def _is_number(tok):
    try:
        float(tok); return True
    except ValueError:
        return False

def _fmt(x):
    return str(int(x)) if float(x).is_integer() else str(x)

def simplify(n):                                        # delete nodes, post-order, O(n)
    if n.is_leaf():
        return ExprNode(n.value)
    L, R, op = simplify(n.left), simplify(n.right), n.value
    lnum = L.is_leaf() and _is_number(L.value)
    rnum = R.is_leaf() and _is_number(R.value)
    if lnum and rnum and not (op == "/" and float(R.value) == 0):
        return ExprNode(_fmt(OPS[op](float(L.value), float(R.value))))   # constant folding
    lv = float(L.value) if lnum else None
    rv = float(R.value) if rnum else None
    if op == "+" and lv == 0: return R
    if op in "+-" and rv == 0: return L
    if op == "*" and (lv == 0 or rv == 0): return ExprNode("0")
    if op == "*" and lv == 1: return R
    if op in "*/" and rv == 1: return L
    if op == "^" and rv == 1: return L
    if op == "^" and rv == 0: return ExprNode("1")
    return ExprNode(op, L, R)

def count_operators(n):
    return 0 if n.is_leaf() else 1 + count_operators(n.left) + count_operators(n.right)

def expr_height(n):
    return 0 if n.is_leaf() else 1 + max(expr_height(n.left), expr_height(n.right))


e = build_from_infix("(x * 1 + 0) * (2 * 3) + y * (z - z * 1) * 0")
banner("Access: root operator and its operands")
print("root =", e.value, "| left =", to_infix(e.left), "| right =", to_infix(e.right))

banner("Search")
print("variables =", sorted(find_variables(e)), "| contains 'y'?", contains(e, "y"),
      "| contains 'w'?", contains(e, "w"))

banner("Delete: simplify")
s = simplify(e)
print("before:", to_infix(e), f"  ({count_operators(e)} operators, height {expr_height(e)})")
print("after :", to_infix(s), f"  ({count_operators(s)} operators, height {expr_height(s)})")

banner("Insert/update: substitute x := (a + 1)")
s2 = substitute(s, "x", build_from_infix("a + 1"))
print(to_infix(s2), "| prefix:", to_prefix(s2))
draw_binary(s2, label=lambda n: n.value)
print("value with a=4:", evaluate(s2, {"a": 4}))


▶ Access: root operator and its operands
root = + | left = (x * 1 + 0) * (2 * 3) | right = y * (z - z * 1) * 0

▶ Search
variables = ['x', 'y', 'z'] | contains 'y'? True | contains 'w'? False

▶ Delete: simplify
before: (x * 1 + 0) * (2 * 3) + y * (z - z * 1) * 0   (9 operators, height 5)
after : x * 6   (1 operators, height 1)

▶ Insert/update: substitute x := (a + 1)
(a + 1) * 6 | prefix: * + a 1 6
*
├── L: +
│   ├── L: a
│   └── R: 1
└── R: 6
value with a=4: 30.0


### Expression tree — complexity ($n$ = number of tokens)

| Operation | Time | Notes |
|---|---|---|
| tokenize | $O(\text{length of string})$ | regex scan |
| infix → postfix (shunting-yard) | $O(n)$ | each operator pushed and popped once |
| build tree from postfix / prefix | $O(n)$ | one stack operation per token |
| to prefix / infix / postfix (traversal) | $O(n)$ | one visit per node |
| evaluate | $O(n)$ | post-order |
| search variable / token | $O(n)$ | |
| substitute (insert subtree) | $O(n + m)$ | $m$ = size of the inserted expression |
| simplify (delete nodes) | $O(n)$ | single post-order pass |
| **Space** | $O(n)$ nodes; $O(h)$ recursion | $h$ = nesting depth |

## 2.3 Trie (prefix tree)

A **trie** stores strings character by character along root-to-node paths. Words that share a prefix share the same path:

```
insert: car, cart, care, cat, dog

          (root)
          /    \
         c      d
         |      |
         a      o
        / \     |
       r●  t●   g●          ● = "a word ends here"
      / \
     t●  e●
```

Key facts:
- The cost of insert/search/delete depends on the **length $L$ of the word**, *not* on the number of stored words $n$ → $O(L)$.
- Prefix queries are natural: "which words start with `car`?" → walk to the node for `car`, then traverse below it. Used in **autocomplete**, spell checking, IP routing, and word games.
- Each node stores `children` (a dict from character to child), `is_end`, and — to answer "how many words start with this prefix?" in $O(L)$ — a `prefix_count`.

In [ ]:
class TrieNode:
    __slots__ = ("children", "is_end", "prefix_count", "value")
    def __init__(self):
        self.children = {}        # char -> TrieNode
        self.is_end = False       # does a word end at this node?
        self.prefix_count = 0     # how many stored words pass through this node
        self.value = None         # optional payload (makes the trie a map)


class Trie:
    def __init__(self, words=()):
        self.root = TrieNode()
        for w in words:
            self.insert(w)

    def __len__(self):
        return self.root.prefix_count

    def _walk(self, s):                                  # follow s from the root, O(len(s))
        node = self.root
        for ch in s:
            node = node.children.get(ch)
            if node is None:
                return None
        return node

    # ---------------------------------------------------------- insert / update
    def insert(self, word, value=None):                  # O(L)
        """Returns True if the word is new; otherwise just updates its value."""
        existing = self._walk(word)
        if existing is not None and existing.is_end:
            existing.value = value
            return False
        node = self.root
        node.prefix_count += 1
        for ch in word:
            node = node.children.setdefault(ch, TrieNode())
            node.prefix_count += 1
        node.is_end = True
        node.value = value
        return True

    # ---------------------------------------------------------- search / access
    def contains(self, word):                            # O(L)
        node = self._walk(word)
        return node is not None and node.is_end

    __contains__ = contains

    def get(self, word, default=None):                   # O(L)
        node = self._walk(word)
        return node.value if node is not None and node.is_end else default

    def starts_with(self, prefix):                       # O(P)
        return self._walk(prefix) is not None

    def count_prefix(self, prefix):                      # O(P)
        node = self._walk(prefix)
        return node.prefix_count if node else 0

    # ---------------------------------------------------------- delete
    def delete(self, word):                              # O(L)
        """Remove a word and prune nodes no longer used by any word."""
        if not self.contains(word):
            return False
        node = self.root
        node.prefix_count -= 1
        for ch in word:
            child = node.children[ch]
            child.prefix_count -= 1
            if child.prefix_count == 0:                  # nobody else uses this branch
                del node.children[ch]
                return True
            node = child
        node.is_end = False
        node.value = None
        return True

    # ---------------------------------------------------------- traversal / specialized
    def _collect(self, node, prefix, out, limit):
        if limit is not None and len(out) >= limit:
            return
        if node.is_end:
            out.append(prefix)
        for ch in sorted(node.children):                 # sorted → lexicographic order
            self._collect(node.children[ch], prefix + ch, out, limit)

    def words(self):                                     # O(total characters)
        out = []
        self._collect(self.root, "", out, None)
        return out

    def autocomplete(self, prefix, limit=None):          # O(P + size of the subtree)
        node = self._walk(prefix)
        out = []
        if node:
            self._collect(node, prefix, out, limit)
        return out

    def longest_common_prefix(self):                     # O(LCP length)
        node, lcp = self.root, ""
        while len(node.children) == 1 and not node.is_end:
            ch, node = next(iter(node.children.items()))
            lcp += ch
        return lcp

    def node_count(self):
        count, st = 0, [self.root]
        while st:
            n = st.pop(); count += 1
            st.extend(n.children.values())
        return count

def draw_trie(node, prefix="", is_root=True, label="(root)"):
    if is_root:
        print(label)
    items = sorted(node.children.items())
    for i, (ch, child) in enumerate(items):
        last = i == len(items) - 1
        print(prefix + ("└── " if last else "├── ") + ch + (" ●" if child.is_end else "")
              + f"  [{child.prefix_count}]")
        draw_trie(child, prefix + ("    " if last else "│   "), False)

In [ ]:
trie = Trie()
banner("Insert: car, cart, care, cat, dog, do  (● = end of word, [k] = words through node)")
for w in ["car", "cart", "care", "cat", "dog", "do"]:
    trie.insert(w)
draw_trie(trie.root)
print("stored words:", len(trie), "| nodes:", trie.node_count())

banner("Search / lookup")
for w in ["car", "ca", "cart", "cars"]:
    print(f"contains({w!r}) = {trie.contains(w)}", end="   ")
print("\nstarts_with('ca') =", trie.starts_with("ca"), "| count_prefix('car') =", trie.count_prefix("car"))

banner("Traversal / specialized: all words (sorted) and autocomplete")
print("words()              :", trie.words())
print("autocomplete('car')  :", trie.autocomplete("car"))
print("autocomplete('d', 1) :", trie.autocomplete("d", limit=1))

banner("Delete: 'cart' (prunes the 't' node), 'car' (only unmarks), 'zebra' (absent)")
print(trie.delete("cart"), trie.delete("car"), trie.delete("zebra"))
draw_trie(trie.root)

banner("Access / update: the trie as a dictionary (word -> value)")
phone = Trie()
phone.insert("alice", "555-0101"); phone.insert("alex", "555-0199"); phone.insert("bob", "555-0144")
phone.insert("alice", "555-0102")          # update existing key
print("get('alice') =", phone.get("alice"), "| get('al') =", phone.get("al"),
      "| contacts starting 'al':", phone.autocomplete("al"))

banner("Longest common prefix")
print(Trie(["interview", "internet", "interval", "internal"]).longest_common_prefix())


▶ Insert: car, cart, care, cat, dog, do  (● = end of word, [k] = words through node)
(root)
├── c  [4]
│   └── a  [4]
│       ├── r ●  [3]
│       │   ├── e ●  [1]
│       │   └── t ●  [1]
│       └── t ●  [1]
└── d  [2]
    └── o ●  [2]
        └── g ●  [1]
stored words: 6 | nodes: 10

▶ Search / lookup
contains('car') = True   contains('ca') = False   contains('cart') = True   contains('cars') = False   
starts_with('ca') = True | count_prefix('car') = 3

▶ Traversal / specialized: all words (sorted) and autocomplete
words()              : ['car', 'care', 'cart', 'cat', 'do', 'dog']
autocomplete('car')  : ['car', 'care', 'cart']
autocomplete('d', 1) : ['do']

▶ Delete: 'cart' (prunes the 't' node), 'car' (only unmarks), 'zebra' (absent)
True True False
(root)
├── c  [2]
│   └── a  [2]
│       ├── r  [1]
│       │   └── e ●  [1]
│       └── t ●  [1]
└── d  [2]
    └── o ●  [2]
        └── g ●  [1]

▶ Access / update: the trie as a dictionary (word -> value)
get('alice') = 555-0102 | 

### Trie vs. a set of strings vs. a sorted list
Searching a word in a trie costs $O(L)$ no matter how many words are stored. Below we compare with a sorted list (binary search: $O(L \log n)$ comparisons) and a Python `set` (hashing: $O(L)$ expected). The trie's real advantage is **prefix** queries, which a hash set can't answer without scanning everything. (In CPython, a trie built from Python objects has a larger constant factor than the C-implemented `set` and `bisect`; a sorted list also answers prefix queries well, but inserting into it costs $O(n)$, while a trie insert stays $O(L)$.)

In [ ]:
import bisect, string
words = {"".join(random.choices(string.ascii_lowercase, k=random.randint(3, 10))) for _ in range(20_000)}
big = Trie(words)
srt = sorted(words)
queries = random.sample(sorted(words), 2000) + ["zzzzq"] * 10

def prefix_scan_set(p):   return [w for w in words if w.startswith(p)]
def prefix_sorted(p):
    i = bisect.bisect_left(srt, p); out = []
    while i < len(srt) and srt[i].startswith(p):
        out.append(srt[i]); i += 1
    return out

print("trie search   (ms):", round(time_op("for q in Q: T.contains(q)", T=big, Q=queries), 2))
print("set search    (ms):", round(time_op("for q in Q: q in S", S=words, Q=queries), 2))
print("prefix 'ab' via trie       (ms):", round(time_op("T.autocomplete('ab')", number=50, T=big), 2))
print("prefix 'ab' via set scan   (ms):", round(time_op("f('ab')", number=50, f=prefix_scan_set), 2))
print("prefix 'ab' via sorted list(ms):", round(time_op("f('ab')", number=50, f=prefix_sorted), 2))
print("same answers:", big.autocomplete("ab") == sorted(prefix_scan_set("ab")) == prefix_sorted("ab"))

trie search   (ms): 1.33
set search    (ms): 0.03


prefix 'ab' via trie       (ms): 1.49


prefix 'ab' via set scan   (ms): 47.45
prefix 'ab' via sorted list(ms): 0.28
same answers: True


### Trie — complexity ($L$ = word length, $P$ = prefix length, $\Sigma$ = alphabet size, $N$ = total characters stored)

| Operation | Time | Notes |
|---|---|---|
| insert / update | $O(L)$ | independent of the number of words $n$ |
| search (exact) | $O(L)$ | |
| `starts_with` / `count_prefix` | $O(P)$ | `prefix_count` makes counting $O(P)$ |
| delete | $O(L)$ | with pruning of unused branches |
| autocomplete | $O(P + k \cdot L)$ | $k$ = number of words returned (sorting children adds $\log \Sigma$) |
| list all words (traversal) | $O(N)$ | lexicographic order for free |
| **Space** | $O(N)$ nodes | each node holds up to $\Sigma$ child links (dict: only used ones) |

## 2.4 Binary Search Tree (BST)

A binary tree with the **BST property**: for every node with key $k$,

$$\text{all keys in the left subtree} < k < \text{all keys in the right subtree}.$$

```
              50
            /    \
          30      70
         /  \    /  \
       20   40  60   80
```

Because of this ordering, **search** discards half of the remaining tree at each step (if the tree is balanced) — exactly like binary search in a sorted array, but with $O(\log n)$ insert/delete too.

**Key fact (structural induction):** the *in-order* traversal of a BST lists the keys in **sorted order**.
*Proof sketch:* by hypothesis, in-order($T_L$) and in-order($T_R$) are sorted; all of $T_L$ < root < all of $T_R$; so in-order($T_L$), root, in-order($T_R$) is sorted.

**The catch:** the shape depends on insertion order. Inserting keys in sorted order produces a **degenerate** tree (a linked list) of height $n-1$, making every operation $O(n)$. Section 3.2 (Red-Black trees) fixes this.

### Deletion — the three cases
1. **Leaf** → just remove it.
2. **One child** → replace the node by its child.
3. **Two children** → copy in the **in-order successor** (the minimum of the right subtree), then delete that successor (which has at most one child, so we fall into case 1 or 2).

In [ ]:
class BSTNode:
    __slots__ = ("key", "value", "left", "right")
    def __init__(self, key, value=None):
        self.key, self.value = key, value
        self.left = self.right = None


class BST:
    def __init__(self, items=()):
        self.root = None
        self._size = 0
        for k in items:
            self.insert(k)

    def __len__(self):
        return self._size

    # ------------------------------------------------------------ search / access
    def search(self, key):                     # O(h)
        node = self.root
        while node:
            if key == node.key:
                return node
            node = node.left if key < node.key else node.right
        return None

    def get(self, key, default=None):          # O(h)
        node = self.search(key)
        return node.value if node else default

    def __contains__(self, key):
        return self.search(key) is not None

    def min(self):                             # O(h): leftmost node
        node = self.root
        if node is None: raise ValueError("empty tree")
        while node.left:
            node = node.left
        return node.key

    def max(self):                             # O(h): rightmost node
        node = self.root
        if node is None: raise ValueError("empty tree")
        while node.right:
            node = node.right
        return node.key

    def successor(self, key):                  # O(h): smallest key > key
        node, best = self.root, None
        while node:
            if key < node.key:
                best, node = node.key, node.left
            else:
                node = node.right
        return best

    def predecessor(self, key):                # O(h): largest key < key
        node, best = self.root, None
        while node:
            if key > node.key:
                best, node = node.key, node.right
            else:
                node = node.left
        return best

    # ------------------------------------------------------------ insert / update
    def insert(self, key, value=None):         # O(h); returns True if the key was new
        if self.root is None:
            self.root = BSTNode(key, value); self._size = 1
            return True
        node = self.root
        while True:
            if key == node.key:                # existing key -> update value
                node.value = value
                return False
            if key < node.key:
                if node.left is None:
                    node.left = BSTNode(key, value); break
                node = node.left
            else:
                if node.right is None:
                    node.right = BSTNode(key, value); break
                node = node.right
        self._size += 1
        return True

    # ------------------------------------------------------------ delete
    def delete(self, key):                     # O(h)
        self.root, removed = self._delete(self.root, key)
        if removed:
            self._size -= 1
        return removed

    def _delete(self, node, key):
        if node is None:
            return None, False
        if key < node.key:
            node.left, removed = self._delete(node.left, key)
            return node, removed
        if key > node.key:
            node.right, removed = self._delete(node.right, key)
            return node, removed
        # found it
        if node.left is None:                  # case 1 or 2
            return node.right, True
        if node.right is None:                 # case 2
            return node.left, True
        succ = node.right                      # case 3: in-order successor
        while succ.left:
            succ = succ.left
        node.key, node.value = succ.key, succ.value
        node.right, _ = self._delete(node.right, succ.key)
        return node, True

    # ------------------------------------------------------------ traversal & specialized
    def inorder(self):                         # sorted keys, O(n), iterative
        st, node = [], self.root
        while st or node:
            while node:
                st.append(node); node = node.left
            node = st.pop()
            yield node.key
            node = node.right

    def range_query(self, lo, hi):             # O(h + k): keys in [lo, hi]
        out = []
        def visit(n):
            if n is None: return
            if lo < n.key: visit(n.left)       # left subtree may contain keys >= lo
            if lo <= n.key <= hi: out.append(n.key)
            if n.key < hi: visit(n.right)
        visit(self.root)
        return out

    def kth_smallest(self, k):                 # O(h + k)
        for i, key in enumerate(self.inorder(), start=1):
            if i == k:
                return key
        raise IndexError("k out of range")

    def height(self):                          # iterative BFS so degenerate trees don't hit recursion limits
        if not self.root: return -1
        h, q = -1, deque([self.root])
        while q:
            h += 1
            for _ in range(len(q)):
                n = q.popleft()
                if n.left: q.append(n.left)
                if n.right: q.append(n.right)
        return h

    def is_valid(self):                        # BST property <=> in-order is strictly increasing
        keys = list(self.inorder())
        return all(a < b for a, b in zip(keys, keys[1:]))

In [ ]:
bst = BST()
banner("Insert: 50 30 70 20 40 60 80 35 45 65")
for k in [50, 30, 70, 20, 40, 60, 80, 35, 45, 65]:
    bst.insert(k)
draw_binary(bst.root)
print("size =", len(bst), "| height =", bst.height(), "| valid BST?", bst.is_valid())

banner("Search / access")
print("search(45) ->", bst.search(45) is not None, "| search(99) ->", bst.search(99))
print("min =", bst.min(), "| max =", bst.max())
print("successor(45) =", bst.successor(45), "| predecessor(60) =", bst.predecessor(60),
      "| successor(80) =", bst.successor(80))

banner("Traversal: in-order is sorted; range query; k-th smallest")
print("in-order:", list(bst.inorder()))
print("range_query(33, 62):", bst.range_query(33, 62), "| 3rd smallest:", bst.kth_smallest(3))

banner("Update: map semantics with values")
bst.insert(40, "forty"); print("get(40) =", bst.get(40))

banner("Delete case 1 (leaf 20), case 2 (one child 60), case 3 (two children 30)")
bst.delete(20); bst.delete(60); bst.delete(30)
draw_binary(bst.root)
print("in-order:", list(bst.inorder()), "| valid?", bst.is_valid())

banner("Delete the root (50) -> replaced by its successor (65)")
bst.delete(50)
draw_binary(bst.root)


▶ Insert: 50 30 70 20 40 60 80 35 45 65
50
├── L: 30
│   ├── L: 20
│   └── R: 40
│       ├── L: 35
│       └── R: 45
└── R: 70
    ├── L: 60
    │   └── R: 65
    └── R: 80
size = 10 | height = 3 | valid BST? True

▶ Search / access
search(45) -> True | search(99) -> None
min = 20 | max = 80
successor(45) = 50 | predecessor(60) = 50 | successor(80) = None

▶ Traversal: in-order is sorted; range query; k-th smallest
in-order: [20, 30, 35, 40, 45, 50, 60, 65, 70, 80]
range_query(33, 62): [35, 40, 45, 50, 60] | 3rd smallest: 35

▶ Update: map semantics with values
get(40) = forty

▶ Delete case 1 (leaf 20), case 2 (one child 60), case 3 (two children 30)
50
├── L: 40
│   ├── L: 35
│   └── R: 45
└── R: 70
    ├── L: 65
    └── R: 80
in-order: [35, 40, 45, 50, 65, 70, 80] | valid? True

▶ Delete the root (50) -> replaced by its successor (65)
65
├── L: 40
│   ├── L: 35
│   └── R: 45
└── R: 70
    └── R: 80


### The degenerate-tree problem — measured

Same keys, different insertion order:

In [ ]:
n = 900
keys = list(range(n))
shuffled = keys[:]; random.shuffle(shuffled)
bst_sorted, bst_random = BST(keys), BST(shuffled)
print(f"n = {n}:  ⌈log2(n+1)⌉-1 = {math.ceil(math.log2(n + 1)) - 1}")
print("height when inserted in sorted order :", bst_sorted.height())
print("height when inserted in random order :", bst_random.height())

probe = random.sample(keys, 300)
t1 = time_op("for k in P: B.search(k)", B=bst_sorted, P=probe)
t2 = time_op("for k in P: B.search(k)", B=bst_random, P=probe)
print(f"300 searches: degenerate tree {t1:.2f} ms vs random tree {t2:.2f} ms  (~{t1 / t2:.0f}x slower)")

tiny = BST([1, 2, 3, 4, 5])
print("\nSorted insert of 1..5 builds a 'linked list':")
draw_binary(tiny.root)

n = 900:  ⌈log2(n+1)⌉-1 = 9
height when inserted in sorted order : 899
height when inserted in random order : 20
300 searches: degenerate tree 7.45 ms vs random tree 0.23 ms  (~33x slower)

Sorted insert of 1..5 builds a 'linked list':
1
└── R: 2
    └── R: 3
        └── R: 4
            └── R: 5


### BST — complexity ($h$ = height)

| Operation | Average (random keys) $h = O(\log n)$ | Worst (degenerate) $h = n-1$ |
|---|---|---|
| search / get / contains | $O(\log n)$ | $O(n)$ |
| insert / update | $O(\log n)$ | $O(n)$ |
| delete | $O(\log n)$ | $O(n)$ |
| min / max / successor / predecessor | $O(\log n)$ | $O(n)$ |
| range query ($k$ results) | $O(\log n + k)$ | $O(n)$ |
| in-order traversal | $O(n)$ | $O(n)$ |
| **Space** | $O(n)$ | $O(n)$ |

All operations are $O(h)$; balancing (Section 3.2) guarantees $h = O(\log n)$.

## 2.5 Priority Queue — Binary Min-Heap and Max-Heap

A **priority queue** returns elements by **priority**, not arrival order: hospital triage, Dijkstra's shortest paths, event simulation, OS schedulers.

A **binary heap** implements it as a **complete binary tree** stored in an **array** (Section 2.1, implementation 3) with the **heap property**:

- **Min-heap:** every parent ≤ its children → the minimum is at the root (`a[0]`).
- **Max-heap:** every parent ≥ its children → the maximum is at the root.

```
min-heap as a tree                 as an array
          1                   index: 0  1  2  3  4  5  6
        /   \                 value: 1  3  2  7  4  5  6
       3     2
      / \   / \
     7   4 5   6
```

Note: a heap is **not** sorted — siblings have no order. It only guarantees the root is the extreme element. That's enough to make **find-min $O(1)$** and **insert / extract-min $O(\log n)$**, since the tree is complete and so has height $\lfloor \log_2 n \rfloor$.

### The two repair operations
- **sift-up** (after inserting at the end): swap with the parent while smaller than the parent.
- **sift-down** (after moving the last element to the root): swap with the smaller child while larger than it.

### Building a heap in $O(n)$ (heapify)
Calling sift-down on nodes $\lfloor n/2 \rfloor - 1, \dots, 0$ builds a heap in $O(n)$ — better than $n$ pushes ($O(n \log n)$) — because most nodes are near the bottom and sift down only a little: $\sum_{h} \frac{n}{2^{h+1}} \cdot O(h) = O(n)$.

In [ ]:
class BinaryHeap:
    """Array-based binary heap. kind='min' (default) or 'max'."""

    def __init__(self, items=(), kind="min"):
        if kind not in ("min", "max"):
            raise ValueError("kind must be 'min' or 'max'")
        self.kind = kind
        self._a = list(items)
        self._heapify()                                  # O(n)

    # x should sit closer to the root than y
    def _before(self, x, y):
        return x < y if self.kind == "min" else x > y

    # ------------------------------------------------------------ repair primitives
    def _sift_up(self, i):                               # O(log n)
        a = self._a
        while i > 0:
            p = (i - 1) // 2
            if self._before(a[i], a[p]):
                a[i], a[p] = a[p], a[i]
                i = p
            else:
                break

    def _sift_down(self, i):                             # O(log n)
        a, n = self._a, len(self._a)
        while True:
            l, r, best = 2 * i + 1, 2 * i + 2, i
            if l < n and self._before(a[l], a[best]): best = l
            if r < n and self._before(a[r], a[best]): best = r
            if best == i:
                return
            a[i], a[best] = a[best], a[i]
            i = best

    def _fix(self, i):                                   # restore order around index i
        if i > 0 and self._before(self._a[i], self._a[(i - 1) // 2]):
            self._sift_up(i)
        else:
            self._sift_down(i)

    def _heapify(self):                                  # O(n) bottom-up construction
        for i in range(len(self._a) // 2 - 1, -1, -1):
            self._sift_down(i)

    # ------------------------------------------------------------ access
    def peek(self):                                      # find-min / find-max, O(1)
        if not self._a:
            raise IndexError("peek from empty heap")
        return self._a[0]

    def __len__(self):
        return len(self._a)

    def as_list(self):                                   # internal array (level order)
        return list(self._a)

    # ------------------------------------------------------------ insert
    def push(self, x):                                   # O(log n)
        self._a.append(x)
        self._sift_up(len(self._a) - 1)

    # ------------------------------------------------------------ delete
    def pop(self):                                       # extract-min / extract-max, O(log n)
        if not self._a:
            raise IndexError("pop from empty heap")
        last = self._a.pop()
        if not self._a:
            return last
        top, self._a[0] = self._a[0], last
        self._sift_down(0)
        return top

    def delete_at(self, i):                              # delete arbitrary position, O(log n)
        removed = self._a[i]
        last = self._a.pop()
        if i < len(self._a):
            self._a[i] = last
            self._fix(i)
        return removed

    # ------------------------------------------------------------ search
    def find(self, x):                                   # O(n): heaps are NOT search structures
        for i, v in enumerate(self._a):
            if v == x:
                return i
        return -1

    def remove(self, x):                                 # O(n) find + O(log n) delete
        i = self.find(x)
        if i == -1:
            raise ValueError(f"{x!r} not in heap")
        return self.delete_at(i)

    # ------------------------------------------------------------ update
    def update_at(self, i, new_value):                   # decrease-key / increase-key, O(log n)
        self._a[i] = new_value
        self._fix(i)

    # ------------------------------------------------------------ specialized
    def pushpop(self, x):                                # push then pop, in one O(log n) step
        if self._a and self._before(self._a[0], x):
            x, self._a[0] = self._a[0], x
            self._sift_down(0)
        return x

    def opposite_extreme(self):                          # max of a min-heap (or min of a max-heap)
        """Only leaves can hold it; leaves are a[n//2:], so this scans n/2 items: O(n)."""
        leaves = self._a[len(self._a) // 2:]
        return max(leaves) if self.kind == "min" else min(leaves)

    def is_valid(self):
        a = self._a
        return all(not self._before(a[i], a[(i - 1) // 2]) for i in range(1, len(a)))

    def __repr__(self):
        return f"{self.kind.capitalize()}Heap({self._a})"


class MinHeap(BinaryHeap):
    def __init__(self, items=()):
        super().__init__(items, "min")

class MaxHeap(BinaryHeap):
    def __init__(self, items=()):
        super().__init__(items, "max")

def draw_heap(h):
    draw_binary(ArrayBinaryTree(h.as_list()).to_linked())

In [ ]:
h = MinHeap()
banner("Insert: push 7, 3, 9, 1, 5, 4, 8, 2  (watch the minimum rise to the root)")
for x in [7, 3, 9, 1, 5, 4, 8, 2]:
    h.push(x)
    print(f"push {x}: array = {h.as_list()}")
draw_heap(h)

banner("Access: peek() = find-min in O(1)")
print("min =", h.peek(), "| size =", len(h), "| heap property holds?", h.is_valid())

banner("Specialized: find-MAX in a min-heap must scan the leaves (O(n))")
print("max =", h.opposite_extreme(), "| leaves:", h.as_list()[len(h) // 2:])

banner("Search: find(5) returns an array index; heaps can't search in O(log n)")
print("index of 5 =", h.find(5), "| index of 42 =", h.find(42))

banner("Delete: pop() = extract-min")
print("popped", h.pop(), "->", h.as_list(), "| valid?", h.is_valid())

banner("Update: decrease-key — change the 9 into 0 (it sifts up to the root)")
h.update_at(h.find(9), 0)
print(h.as_list(), "| min =", h.peek(), "| valid?", h.is_valid())

banner("Delete an arbitrary element: remove(5)")
h.remove(5)
print(h.as_list(), "| valid?", h.is_valid())

banner("Traversal in priority order (destructive): pop until empty")
print([h.pop() for _ in range(len(h))])

banner("Max-heap built in O(n) with heapify")
mx = MaxHeap([3, 1, 6, 5, 2, 4])
print(mx, "| max =", mx.peek(), "| min (scan leaves) =", mx.opposite_extreme())
draw_heap(mx)


▶ Insert: push 7, 3, 9, 1, 5, 4, 8, 2  (watch the minimum rise to the root)
push 7: array = [7]
push 3: array = [3, 7]
push 9: array = [3, 7, 9]
push 1: array = [1, 3, 9, 7]
push 5: array = [1, 3, 9, 7, 5]
push 4: array = [1, 3, 4, 7, 5, 9]
push 8: array = [1, 3, 4, 7, 5, 9, 8]
push 2: array = [1, 2, 4, 3, 5, 9, 8, 7]
1
├── L: 2
│   ├── L: 3
│   │   └── L: 7
│   └── R: 5
└── R: 4
    ├── L: 9
    └── R: 8

▶ Access: peek() = find-min in O(1)
min = 1 | size = 8 | heap property holds? True

▶ Specialized: find-MAX in a min-heap must scan the leaves (O(n))
max = 9 | leaves: [5, 9, 8, 7]

▶ Search: find(5) returns an array index; heaps can't search in O(log n)
index of 5 = 4 | index of 42 = -1

▶ Delete: pop() = extract-min
popped 1 -> [2, 3, 4, 7, 5, 9, 8] | valid? True

▶ Update: decrease-key — change the 9 into 0 (it sifts up to the root)
[0, 3, 2, 7, 5, 4, 8] | min = 0 | valid? True

▶ Delete an arbitrary element: remove(5)
[0, 3, 2, 7, 8, 4] | valid? True

▶ Traversal in priority ord

### Applications: heapsort, top-$k$, and a task scheduler

In [ ]:
def heap_sort(items):                     # O(n log n), uses heapify + n pops
    h = MinHeap(items)
    return [h.pop() for _ in range(len(h))]

def k_largest(stream, k):                 # O(n log k) time, O(k) space: keep a MIN-heap of size k
    h = MinHeap()
    for x in stream:
        if len(h) < k:
            h.push(x)
        else:
            h.pushpop(x)                  # drop the smallest of the k+1 candidates
    return sorted(h.as_list(), reverse=True)

data = [random.randint(0, 999) for _ in range(20)]
print("data        :", data)
print("heap_sort   :", heap_sort(data))
print("k_largest(4):", k_largest(data, 4), "==", sorted(data, reverse=True)[:4])

banner("Task scheduler: (priority, arrival order, task); lower number = more urgent")
import itertools
counter = itertools.count()               # tie-breaker keeps equal priorities in FIFO order
tasks = MinHeap()
for prio, name in [(3, "write report"), (1, "fix prod outage"), (2, "review PR"),
                   (1, "reply to CEO"), (3, "lunch")]:
    tasks.push((prio, next(counter), name))
while tasks:
    prio, _, name = tasks.pop()
    print(f"  [{prio}] {name}")

banner("heapify (O(n)) vs n pushes (O(n log n)) — 200,000 items in DESCENDING order")
# Descending input is the worst case for pushes into a min-heap: every new item sifts all the way up.
# (On random input a push moves up only O(1) levels on average, so the gap is much smaller.)
big = sorted((random.random() for _ in range(200_000)), reverse=True)
print("heapify  (ms):", round(time_op("MinHeap(D)", MinHeap=MinHeap, D=big), 1))
print("n pushes (ms):", round(time_op("h = MinHeap()\nfor x in D: h.push(x)", MinHeap=MinHeap, D=big), 1))

banner("Python's built-in: heapq (a min-heap on a plain list)")
import heapq
pq = [5, 1, 8, 3]
heapq.heapify(pq); heapq.heappush(pq, 0)
print("heapq:", pq, "| pop ->", heapq.heappop(pq), "| nsmallest(2) ->", heapq.nsmallest(2, pq))
neg = [-x for x in [5, 1, 8]]; heapq.heapify(neg)
print("max-heap trick: store negatives -> max =", -heapq.heappop(neg))

data        : [228, 519, 73, 443, 461, 839, 757, 610, 120, 447, 780, 716, 105, 711, 934, 578, 57, 756, 495, 441]
heap_sort   : [57, 73, 105, 120, 228, 441, 443, 447, 461, 495, 519, 578, 610, 711, 716, 756, 757, 780, 839, 934]
k_largest(4): [934, 839, 780, 757] == [934, 839, 780, 757]

▶ Task scheduler: (priority, arrival order, task); lower number = more urgent
  [1] fix prod outage
  [1] reply to CEO
  [2] review PR
  [3] write report
  [3] lunch

▶ heapify (O(n)) vs n pushes (O(n log n)) — 200,000 items in DESCENDING order


heapify  (ms): 107.9


n pushes (ms): 523.4

▶ Python's built-in: heapq (a min-heap on a plain list)
heapq: [1, 3, 8, 5] | pop -> 0 | nsmallest(2) -> [1, 3]
max-heap trick: store negatives -> max = 8


### Binary heap — complexity

| Operation | Min-heap | Max-heap | Notes |
|---|---|---|---|
| peek (find-min / find-max) | $O(1)$ | $O(1)$ | root of the tree, `a[0]` |
| find the *other* extreme | $O(n)$ | $O(n)$ | scan the $\lceil n/2 \rceil$ leaves |
| push (insert) | $O(\log n)$ | $O(\log n)$ | sift-up |
| pop (extract-min / extract-max) | $O(\log n)$ | $O(\log n)$ | sift-down |
| search arbitrary value | $O(n)$ | $O(n)$ | no ordering between siblings |
| delete at known index | $O(\log n)$ | $O(\log n)$ | swap with last + fix |
| update (decrease/increase-key) at index | $O(\log n)$ | $O(\log n)$ | sift-up or sift-down |
| build from $n$ items (heapify) | $O(n)$ | $O(n)$ | bottom-up |
| heapsort | $O(n \log n)$ | $O(n \log n)$ | |
| **Space** | $O(n)$, no pointers | $O(n)$ | array storage |

---
# Part 3 — Advanced Data Structures

## 3.1 Disjoint Sets (Union–Find)

Maintain a collection of **non-overlapping groups** under two operations:
- **find(x)** — which group is $x$ in? (returns the group's *representative*)
- **union(x, y)** — merge the groups of $x$ and $y$.

Typical questions: *Are these two computers on the same network? Are these two pixels in the same region? Would adding this edge create a cycle?* (Kruskal's minimum spanning tree.)

### Representation: a forest of parent pointers
Each group is a tree; each element points to its parent; the **root is the representative**.

```
groups {a, b, c, d}, {e, f}

     a          e
    / \         |
   b   c        f
       |
       d          find(d) = a,  find(f) = e
```

### Two optimizations that make it almost $O(1)$
1. **Union by rank:** hang the *shorter* tree under the taller one (rank ≈ height upper bound), so trees stay $O(\log n)$ tall.
2. **Path compression:** during `find`, point every node on the path directly at the root, flattening the tree for future queries.

With both, a sequence of $m$ operations on $n$ elements costs $O(m\,\alpha(n))$, where $\alpha$ is the **inverse Ackermann function** — $\alpha(n) \le 4$ for any $n$ smaller than the number of atoms in the universe. Effectively constant.

First, a naive "quick-find" version for contrast: every element stores its group id directly, so `find` is $O(1)$ but `union` must relabel a whole group — $O(n)$.

In [ ]:
class QuickFind:
    """Naive version: find O(1), union O(n)."""
    def __init__(self, elements):
        self.id = {e: e for e in elements}
    def find(self, x):
        return self.id[x]
    def union(self, a, b):
        ida, idb = self.id[a], self.id[b]
        if ida == idb:
            return False
        for e in self.id:                    # relabel every member of b's group: O(n)
            if self.id[e] == idb:
                self.id[e] = ida
        return True


class DisjointSet:
    """Union by rank + path compression: ~O(1) amortized per operation."""

    def __init__(self, elements=()):
        self.parent = {}
        self.rank = {}
        self._size = {}             # size of each set, stored at its root
        self.count = 0              # number of disjoint sets
        for e in elements:
            self.make_set(e)

    # ---------------------------------------------------------- insert
    def make_set(self, x):                       # O(1)
        if x in self.parent:
            return False
        self.parent[x] = x
        self.rank[x] = 0
        self._size[x] = 1
        self.count += 1
        return True

    # ---------------------------------------------------------- access / search
    def find(self, x):                           # O(α(n)) amortized
        if x not in self.parent:
            raise KeyError(x)
        root = x
        while self.parent[root] != root:         # 1st pass: locate the root
            root = self.parent[root]
        while self.parent[x] != root:            # 2nd pass: path compression
            nxt = self.parent[x]
            self.parent[x] = root
            x = nxt
        return root

    def connected(self, a, b):                   # O(α(n))
        return self.find(a) == self.find(b)

    def set_size(self, x):                       # O(α(n))
        return self._size[self.find(x)]

    def __contains__(self, x):
        return x in self.parent

    def __len__(self):
        return len(self.parent)

    # ---------------------------------------------------------- union (specialized)
    def union(self, a, b):                       # O(α(n))
        ra, rb = self.find(a), self.find(b)
        if ra == rb:
            return False                         # already in the same set
        if self.rank[ra] < self.rank[rb]:        # union by rank: attach shorter under taller
            ra, rb = rb, ra
        self.parent[rb] = ra
        self._size[ra] += self._size[rb]
        if self.rank[ra] == self.rank[rb]:
            self.rank[ra] += 1
        self.count -= 1
        return True

    # ---------------------------------------------------------- traversal
    def groups(self):                            # O(n α(n))
        out = defaultdict(list)
        for x in self.parent:
            out[self.find(x)].append(x)
        return dict(out)

In [ ]:
ds = DisjointSet()
banner("Insert: make_set for a..h")
for x in "abcdefgh":
    ds.make_set(x)
print("sets:", ds.count, "|", ds.groups())

banner("Union (specialized): a-b, c-d, b-d, e-f, g-h, f-h")
for x, y in ["ab", "cd", "bd", "ef", "gh", "fh"]:
    ds.union(x, y)
print("sets:", ds.count, "|", ds.groups())
print("union(a, c) again ->", ds.union("a", "c"), "(already connected)")

banner("Access / search: find returns the representative")
print({x: ds.find(x) for x in "abcdefgh"})
print("connected(a, d)?", ds.connected("a", "d"), "| connected(a, e)?", ds.connected("a", "e"))
print("size of a's set =", ds.set_size("a"))

banner("Path compression in action: build a chain manually, then call find")
chain = DisjointSet(range(6))
for i in range(5):
    chain.parent[i] = i + 1        # 0 -> 1 -> 2 -> 3 -> 4 -> 5 (a worst-case path)
print("before find(0):", chain.parent)
chain.find(0)
print("after  find(0):", chain.parent, " <- every node now points straight at the root 5")


▶ Insert: make_set for a..h
sets: 8 | {'a': ['a'], 'b': ['b'], 'c': ['c'], 'd': ['d'], 'e': ['e'], 'f': ['f'], 'g': ['g'], 'h': ['h']}

▶ Union (specialized): a-b, c-d, b-d, e-f, g-h, f-h
sets: 2 | {'a': ['a', 'b', 'c', 'd'], 'e': ['e', 'f', 'g', 'h']}
union(a, c) again -> False (already connected)

▶ Access / search: find returns the representative
{'a': 'a', 'b': 'a', 'c': 'a', 'd': 'a', 'e': 'e', 'f': 'e', 'g': 'e', 'h': 'e'}
connected(a, d)? True | connected(a, e)? False
size of a's set = 4

▶ Path compression in action: build a chain manually, then call find
before find(0): {0: 1, 1: 2, 2: 3, 3: 4, 4: 5, 5: 5}
after  find(0): {0: 5, 1: 5, 2: 5, 3: 5, 4: 5, 5: 5}  <- every node now points straight at the root 5


### Delete in a disjoint-set forest

Plain union–find has **no delete**: other elements may point *through* $x$ to reach the root, so we cannot simply unlink it.
The standard trick is **lazy deletion with indirection**: users refer to elements through a map `element → internal id`. To delete $x$ we:
1. decrement the live size stored at its root (and drop the set count if it becomes empty),
2. forget the mapping, so the old internal node stays only as a "ghost" path node.

Re-inserting $x$ later gives it a brand-new internal id. All operations keep their $O(\alpha(n))$ cost.

In [ ]:
class DeletableDisjointSet:
    def __init__(self, elements=()):
        self._ds = DisjointSet()
        self._id = {}
        self._next = 0
        for e in elements:
            self.make_set(e)

    def make_set(self, x):
        if x in self._id:
            return False
        self._id[x] = self._next
        self._ds.make_set(self._next)
        self._next += 1
        return True

    def find(self, x):            return self._ds.find(self._id[x])
    def union(self, a, b):        return self._ds.union(self._id[a], self._id[b])
    def connected(self, a, b):    return self.find(a) == self.find(b)
    def set_size(self, x):        return self._ds.set_size(self._id[x])

    @property
    def count(self):
        return self._ds.count

    def delete(self, x):                          # O(α(n))
        root = self.find(x)
        self._ds._size[root] -= 1
        if self._ds._size[root] == 0:
            self._ds.count -= 1
        del self._id[x]

    def groups(self):
        out = defaultdict(list)
        for x in self._id:
            out[self.find(x)].append(x)
        return sorted(out.values())

dds = DeletableDisjointSet("pqrst")
dds.union("p", "q"); dds.union("q", "r"); dds.union("s", "t")
print("groups:", dds.groups(), "| count =", dds.count, "| size(p) =", dds.set_size("p"))
dds.delete("q")                                  # q was the middle of the p-q-r chain
print("after delete(q):", dds.groups(), "| p~r still connected?", dds.connected("p", "r"),
      "| size(p) =", dds.set_size("p"))
dds.delete("s"); dds.delete("t")
print("after deleting s and t:", dds.groups(), "| count =", dds.count)

groups: [['p', 'q', 'r'], ['s', 't']] | count = 2 | size(p) = 3
after delete(q): [['p', 'r'], ['s', 't']] | p~r still connected? True | size(p) = 2
after deleting s and t: [['p', 'r']] | count = 1


### Application: Kruskal's minimum spanning tree

Sort edges by weight; add an edge unless its endpoints are already connected (that would form a cycle). Union–find answers "already connected?" in near-constant time, so Kruskal runs in $O(E \log E)$ — dominated by sorting.

In [ ]:
def kruskal(vertices, edges):
    ds = DisjointSet(vertices)
    mst, total = [], 0
    for w, u, v in sorted(edges):
        if ds.union(u, v):                 # False means u and v were already connected
            mst.append((u, v, w))
            total += w
    return mst, total

V = "ABCDEF"
E = [(4, "A", "B"), (2, "A", "C"), (5, "B", "C"), (10, "B", "D"), (3, "C", "E"),
     (4, "E", "D"), (11, "D", "F"), (6, "E", "F"), (7, "A", "F")]
mst, total = kruskal(V, E)
print("MST edges:", mst, "| total weight =", total)

banner("QuickFind vs DisjointSet: 5,000 elements, 5,000 random unions")
n = 5_000
pairs = [(random.randrange(n), random.randrange(n)) for _ in range(n)]
def run(cls):
    d = cls(range(n))
    for a, b in pairs:
        d.union(a, b)
print("QuickFind   (ms):", round(time_op("run(QF)", run=run, QF=QuickFind), 1))
print("DisjointSet (ms):", round(time_op("run(DS)", run=run, DS=DisjointSet), 1))

MST edges: [('A', 'C', 2), ('C', 'E', 3), ('A', 'B', 4), ('E', 'D', 4), ('E', 'F', 6)] | total weight = 19

▶ QuickFind vs DisjointSet: 5,000 elements, 5,000 random unions


QuickFind   (ms): 812.7
DisjointSet (ms): 4.4


### Disjoint sets — complexity ($\alpha$ = inverse Ackermann, $\le 4$ in practice)

| Operation | Quick-find | Union by rank only | Rank + path compression |
|---|---|---|---|
| make_set (insert) | $O(1)$ | $O(1)$ | $O(1)$ |
| find (access representative) | $O(1)$ | $O(\log n)$ | $O(\alpha(n))$ amortized |
| connected (search) | $O(1)$ | $O(\log n)$ | $O(\alpha(n))$ amortized |
| union | $O(n)$ | $O(\log n)$ | $O(\alpha(n))$ amortized |
| delete (lazy, with indirection) | — | $O(\log n)$ | $O(\alpha(n))$ amortized |
| list all groups (traversal) | $O(n)$ | $O(n \log n)$ | $O(n\,\alpha(n))$ |
| **Space** | $O(n)$ | $O(n)$ | $O(n)$ |

## 3.2 Balanced Binary Trees: the Red-Black Tree

A plain BST can degenerate to height $n-1$ (Section 2.4). A **red-black tree** is a BST whose nodes carry one extra bit — a **color** — and that obeys five rules:

1. Every node is **red** or **black**.
2. The **root** is black.
3. Every leaf (the `NIL` sentinel standing for "no child") is black.
4. A red node has **no red child** (no two reds in a row).
5. For every node, all paths down to `NIL` leaves contain the **same number of black nodes** (the node's *black-height*).

### Why that guarantees $O(\log n)$ height
By rule 5 every root-to-NIL path has the same number $b$ of black nodes; by rule 4 at most every other node on a path is red, so **the longest path is at most twice the shortest**. A structural induction shows a subtree with black-height $b$ has at least $2^{b} - 1$ internal nodes, which gives

$$h \le 2 \log_2(n + 1).$$

### How balance is restored: rotations + recoloring
A **rotation** is a local $O(1)$ pointer change that preserves the BST order:

```
        x                          y
       / \     left_rotate(x)     / \
      a   y    ─────────────►    x   c
         / \   ◄─────────────   / \
        b   c  right_rotate(y) a   b
```

- **Insert:** do a normal BST insert, color the new node **red** (so black-heights don't change), then fix a possible *red–red* violation walking up: **case 1** red uncle → recolor and move up; **case 2/3** black uncle → one or two rotations and stop. At most **2 rotations**.
- **Delete:** do a normal BST delete; if a **black** node was removed, one path lost a black node ("double black"), which is fixed with 4 cases using the sibling. At most **3 rotations**.

We follow the CLRS (Cormen et al.) formulation with a single shared black `NIL` sentinel. The class also has two empty "hook" methods, `_pull` and `_pull_upward`; they do nothing here but let the **Interval Tree** in Section 3.3 keep extra data up to date during rotations.

In [ ]:
RED, BLACK = "R", "B"

class RBNode:
    def __init__(self, key, value=None, color=RED, nil=None):
        self.key, self.value, self.color = key, value, color
        self.left = self.right = self.parent = nil


class RedBlackTree:
    def __init__(self, items=()):
        self.NIL = RBNode(None, color=BLACK)
        self.NIL.left = self.NIL.right = self.NIL.parent = self.NIL
        self.root = self.NIL
        self._size = 0
        for k in items:
            self.insert(k)

    def __len__(self):
        return self._size

    # ---------- augmentation hooks (no-ops here; used by IntervalTree) ----------
    def _new_node(self, key, value):
        return RBNode(key, value, RED, self.NIL)
    def _pull(self, node):
        pass
    def _pull_upward(self, node):
        pass

    # ============================================================ rotations O(1)
    def _left_rotate(self, x):
        y = x.right
        x.right = y.left
        if y.left is not self.NIL:
            y.left.parent = x
        y.parent = x.parent
        if x.parent is self.NIL:
            self.root = y
        elif x is x.parent.left:
            x.parent.left = y
        else:
            x.parent.right = y
        y.left = x
        x.parent = y
        self._pull(x); self._pull(y)

    def _right_rotate(self, y):
        x = y.left
        y.left = x.right
        if x.right is not self.NIL:
            x.right.parent = y
        x.parent = y.parent
        if y.parent is self.NIL:
            self.root = x
        elif y is y.parent.right:
            y.parent.right = x
        else:
            y.parent.left = x
        x.right = y
        y.parent = x
        self._pull(y); self._pull(x)

    # ============================================================ search / access
    def _find_node(self, key):                       # O(log n)
        node = self.root
        while node is not self.NIL and key != node.key:
            node = node.left if key < node.key else node.right
        return node

    def search(self, key):
        node = self._find_node(key)
        return None if node is self.NIL else node

    def get(self, key, default=None):
        node = self._find_node(key)
        return default if node is self.NIL else node.value

    def __contains__(self, key):
        return self._find_node(key) is not self.NIL

    def _min_node(self, node):
        while node.left is not self.NIL:
            node = node.left
        return node

    def _max_node(self, node):
        while node.right is not self.NIL:
            node = node.right
        return node

    def min(self):
        if self.root is self.NIL: raise ValueError("empty tree")
        return self._min_node(self.root).key

    def max(self):
        if self.root is self.NIL: raise ValueError("empty tree")
        return self._max_node(self.root).key

    def successor(self, key):                         # uses parent pointers, O(log n)
        node = self._find_node(key)
        if node is self.NIL:
            raise KeyError(key)
        if node.right is not self.NIL:
            return self._min_node(node.right).key
        p = node.parent
        while p is not self.NIL and node is p.right:
            node, p = p, p.parent
        return None if p is self.NIL else p.key

    # ============================================================ insert O(log n)
    def insert(self, key, value=None):
        """Insert a new key (returns True) or update an existing key's value (returns False)."""
        parent, node = self.NIL, self.root
        while node is not self.NIL:                   # 1) ordinary BST descent
            parent = node
            if key == node.key:
                node.value = value
                return False
            node = node.left if key < node.key else node.right
        z = self._new_node(key, value)                # 2) attach a RED node
        z.parent = parent
        if parent is self.NIL:
            self.root = z
        elif key < parent.key:
            parent.left = z
        else:
            parent.right = z
        self._size += 1
        self._pull_upward(z)
        self._insert_fixup(z)                         # 3) repair red-red violations
        return True

    def _insert_fixup(self, z):
        while z.parent.color == RED:
            gp = z.parent.parent
            if z.parent is gp.left:
                uncle = gp.right
                if uncle.color == RED:                # case 1: recolor, move up
                    z.parent.color = BLACK
                    uncle.color = BLACK
                    gp.color = RED
                    z = gp
                else:
                    if z is z.parent.right:           # case 2: zig-zag -> zig-zig
                        z = z.parent
                        self._left_rotate(z)
                    z.parent.color = BLACK            # case 3: rotate grandparent
                    z.parent.parent.color = RED
                    self._right_rotate(z.parent.parent)
            else:                                     # mirror image
                uncle = gp.left
                if uncle.color == RED:
                    z.parent.color = BLACK
                    uncle.color = BLACK
                    gp.color = RED
                    z = gp
                else:
                    if z is z.parent.left:
                        z = z.parent
                        self._right_rotate(z)
                    z.parent.color = BLACK
                    z.parent.parent.color = RED
                    self._left_rotate(z.parent.parent)
        self.root.color = BLACK

    # ============================================================ delete O(log n)
    def _transplant(self, u, v):                      # replace subtree u by subtree v
        if u.parent is self.NIL:
            self.root = v
        elif u is u.parent.left:
            u.parent.left = v
        else:
            u.parent.right = v
        v.parent = u.parent

    def delete(self, key):
        z = self._find_node(key)
        if z is self.NIL:
            return False
        y, y_original_color = z, z.color
        if z.left is self.NIL:                        # at most one child
            x = z.right
            self._transplant(z, z.right)
        elif z.right is self.NIL:
            x = z.left
            self._transplant(z, z.left)
        else:                                         # two children: use the successor y
            y = self._min_node(z.right)
            y_original_color = y.color
            x = y.right
            if y.parent is z:
                x.parent = y
            else:
                self._transplant(y, y.right)
                y.right = z.right
                y.right.parent = y
            self._transplant(z, y)
            y.left = z.left
            y.left.parent = y
            y.color = z.color
        self._size -= 1
        self._pull_upward(x.parent)
        if y_original_color == BLACK:                 # a black node left some path
            self._delete_fixup(x)
        return True

    def _delete_fixup(self, x):
        while x is not self.root and x.color == BLACK:
            if x is x.parent.left:
                w = x.parent.right                    # sibling
                if w.color == RED:                    # case 1: make the sibling black
                    w.color = BLACK
                    x.parent.color = RED
                    self._left_rotate(x.parent)
                    w = x.parent.right
                if w.left.color == BLACK and w.right.color == BLACK:
                    w.color = RED                     # case 2: push the problem up
                    x = x.parent
                else:
                    if w.right.color == BLACK:        # case 3: make the far nephew red
                        w.left.color = BLACK
                        w.color = RED
                        self._right_rotate(w)
                        w = x.parent.right
                    w.color = x.parent.color          # case 4: rotate and finish
                    x.parent.color = BLACK
                    w.right.color = BLACK
                    self._left_rotate(x.parent)
                    x = self.root
            else:                                     # mirror image
                w = x.parent.left
                if w.color == RED:
                    w.color = BLACK
                    x.parent.color = RED
                    self._right_rotate(x.parent)
                    w = x.parent.left
                if w.right.color == BLACK and w.left.color == BLACK:
                    w.color = RED
                    x = x.parent
                else:
                    if w.left.color == BLACK:
                        w.right.color = BLACK
                        w.color = RED
                        self._left_rotate(w)
                        w = x.parent.left
                    w.color = x.parent.color
                    x.parent.color = BLACK
                    w.left.color = BLACK
                    self._right_rotate(x.parent)
                    x = self.root
        x.color = BLACK

    # ============================================================ traversal & checks
    def inorder(self):                                # O(n), iterative
        st, node = [], self.root
        while st or node is not self.NIL:
            while node is not self.NIL:
                st.append(node); node = node.left
            node = st.pop()
            yield node.key
            node = node.right

    def height(self):
        def h(n):
            return -1 if n is self.NIL else 1 + max(h(n.left), h(n.right))
        return h(self.root)

    def black_height(self):
        b, node = 0, self.root
        while node is not self.NIL:
            b += node.color == BLACK
            node = node.left
        return b

    def validate(self):
        """Assert all five red-black properties + BST order + parent links."""
        assert self.root.color == BLACK, "rule 2: root must be black"
        assert self.NIL.color == BLACK, "rule 3: NIL must be black"
        def check(n, lo, hi):
            if n is self.NIL:
                return 1
            assert (lo is None or lo < n.key) and (hi is None or n.key < hi), "BST order violated"
            if n.color == RED:
                assert n.left.color == BLACK and n.right.color == BLACK, "rule 4: red node with red child"
            for c in (n.left, n.right):
                if c is not self.NIL:
                    assert c.parent is n, "broken parent pointer"
            lb, rb = check(n.left, lo, n.key), check(n.right, n.key, hi)
            assert lb == rb, "rule 5: unequal black-heights"
            return lb + (n.color == BLACK)
        check(self.root, None, None)
        assert sum(1 for _ in self.inorder()) == self._size
        return True

    def draw(self):
        draw_binary(self.root, label=lambda n: f"{n.key} ({'B' if n.color == BLACK else 'R'})",
                    is_empty=lambda n: n is self.NIL)

In [ ]:
rb = RedBlackTree()
banner("Insert 1..10 in SORTED order (the worst case for a plain BST)")
for k in range(1, 11):
    rb.insert(k)
rb.draw()
print(f"size = {len(rb)} | height = {rb.height()} (plain BST would be 9) "
      f"| black-height = {rb.black_height()} | valid? {rb.validate()}")

banner("Search / access")
print("10 in rb?", 10 in rb, "| 42 in rb?", 42 in rb, "| min =", rb.min(), "| max =", rb.max(),
      "| successor(4) =", rb.successor(4))

banner("Update: map semantics")
rb.insert(7, "seven")
print("get(7) =", rb.get(7), "| size unchanged:", len(rb))

banner("Delete 4 (internal node with two children) and 1, 2 (forces fix-up rotations)")
for k in (4, 1, 2):
    rb.delete(k)
rb.draw()
print("in-order:", list(rb.inorder()), "| valid?", rb.validate())


▶ Insert 1..10 in SORTED order (the worst case for a plain BST)
4 (B)
├── L: 2 (B)
│   ├── L: 1 (B)
│   └── R: 3 (B)
└── R: 6 (B)
    ├── L: 5 (B)
    └── R: 8 (R)
        ├── L: 7 (B)
        └── R: 9 (B)
            └── R: 10 (R)
size = 10 | height = 4 (plain BST would be 9) | black-height = 3 | valid? True

▶ Search / access
10 in rb? True | 42 in rb? False | min = 1 | max = 10 | successor(4) = 5

▶ Update: map semantics
get(7) = seven | size unchanged: 10

▶ Delete 4 (internal node with two children) and 1, 2 (forces fix-up rotations)
5 (B)
├── L: 3 (B)
└── R: 8 (R)
    ├── L: 6 (B)
    │   └── R: 7 (R)
    └── R: 9 (B)
        └── R: 10 (R)
in-order: [3, 5, 6, 7, 8, 9, 10] | valid? True


### Stress test and comparison with the plain BST

We perform thousands of random inserts and deletes, validating every red-black property along the way, and then compare heights against the unbalanced BST from Section 2.4.

In [ ]:
rb, ref = RedBlackTree(), set()
for step in range(6000):
    k = random.randrange(2000)
    if random.random() < 0.6:
        rb.insert(k); ref.add(k)
    else:
        assert rb.delete(k) == (k in ref); ref.discard(k)
    if step % 500 == 0:
        rb.validate()
rb.validate()
print("6000 random ops: contents match a Python set?", list(rb.inorder()) == sorted(ref),
      "| all RB properties hold ✓")

print(f"\n{'n':>7} | {'2·log2(n+1)':>11} | {'RB height (sorted input)':>24} | {'BST height (sorted input)':>25}")
for n in [100, 500, 900]:
    rbt, b = RedBlackTree(range(n)), BST(range(n))
    print(f"{n:>7} | {2 * math.log2(n + 1):>11.1f} | {rbt.height():>24} | {b.height():>25}")

n = 900
rbt, b = RedBlackTree(range(n)), BST(range(n))
probe = random.sample(range(n), 300)
t_rb = time_op("for k in P: T.search(k)", T=rbt, P=probe)
t_bs = time_op("for k in P: T.search(k)", T=b, P=probe)
print(f"\n300 searches after sorted inserts: RB {t_rb:.2f} ms vs plain BST {t_bs:.2f} ms")

6000 random ops: contents match a Python set? True | all RB properties hold ✓

      n | 2·log2(n+1) | RB height (sorted input) | BST height (sorted input)
    100 |        13.3 |                       10 |                        99
    500 |        17.9 |                       14 |                       499
    900 |        19.6 |                       16 |                       899

300 searches after sorted inserts: RB 0.20 ms vs plain BST 7.83 ms


### Red-Black tree — complexity

| Operation | Time (worst case) | Notes |
|---|---|---|
| search / get / contains | $O(\log n)$ | height $\le 2\log_2(n+1)$ |
| min / max / successor | $O(\log n)$ | |
| insert / update | $O(\log n)$ | $\le 2$ rotations + $O(\log n)$ recolorings |
| delete | $O(\log n)$ | $\le 3$ rotations |
| rotation | $O(1)$ | a constant number of pointer changes |
| in-order traversal | $O(n)$ | sorted order |
| **Space** | $O(n)$ | + 1 color bit and a parent pointer per node |

**Comparison:** AVL trees are *more strictly* balanced (height $\le 1.44 \log_2 n$), so lookups are slightly faster, but they may rotate more on updates. Red-black trees power `std::map` (C++), `TreeMap` (Java) and the Linux kernel scheduler.

## 3.3 Interval Tree

Store intervals $[\text{low}, \text{high}]$ (meetings, reserved time slots, genome regions, memory ranges) and answer:

> **Which stored intervals overlap a query interval $[a, b]$?**

Two closed intervals overlap iff $\text{low}_1 \le \text{high}_2$ **and** $\text{low}_2 \le \text{high}_1$.

### Design (CLRS "augmenting a data structure")
1. **Underlying structure:** a red-black tree keyed by the interval's **low** endpoint (we use the pair `(low, high)` so equal lows are still ordered).
2. **Extra field:** each node stores `max` = the largest **high** endpoint anywhere in its subtree.
3. **Maintaining it:** `node.max = max(node.high, left.max, right.max)` depends only on the node and its children, so it can be recomputed in $O(1)$ after each rotation and along the $O(\log n)$ path changed by an insert/delete. We plug this into the hooks `_pull` / `_pull_upward` of `RedBlackTree` — all balancing logic is inherited unchanged.
4. **New operation — overlap search in $O(\log n)$:** at node $x$, if $x$'s interval doesn't overlap the query:
   - if `x.left.max ≥ a`, **go left** — if any interval overlaps, one exists on the left (proof below);
   - otherwise **go right** — nothing on the left can reach $a$.

*Why going left is safe:* if `x.left.max ≥ a`, some left interval $i$ has $\text{high}_i \ge a$. If $i$ doesn't overlap, then $\text{low}_i > b$; all right-subtree intervals have even larger lows, so they don't overlap either. Hence no overlap anywhere unless there is one on the left.

```
key = low endpoint, [max] = largest high in subtree

                  [16,21] max 30
                 /               \
        [8,9] max 23            [25,30] max 30
        /        \               /         \
 [5,8] max 10  [15,23] max 23  [17,19] max 20  [26,26] max 26
   ...
```

In [ ]:
class IntervalTree(RedBlackTree):
    """Red-black tree of closed intervals, augmented with subtree max endpoints."""

    def __init__(self, intervals=()):
        super().__init__()
        self.NIL.max = float("-inf")
        for lo, hi in intervals:
            self.add(lo, hi)

    # ------------------------------------------------ augmentation hooks
    def _new_node(self, key, value):
        node = super()._new_node(key, value)
        node.max = key[1]
        return node

    def _pull(self, node):                       # O(1): recompute from children
        node.max = max(node.key[1], node.left.max, node.right.max)

    def _pull_upward(self, node):                # O(log n): fix the path to the root
        while node is not self.NIL:
            self._pull(node)
            node = node.parent

    # ------------------------------------------------ insert / delete / exact search
    def add(self, low, high, value=None):        # O(log n)
        if low > high:
            raise ValueError("low must be <= high")
        return self.insert((low, high), value)

    def remove(self, low, high):                 # O(log n)
        return self.delete((low, high))

    def contains_interval(self, low, high):      # O(log n): exact match
        return (low, high) in self

    # ------------------------------------------------ overlap queries
    @staticmethod
    def overlaps(a, b):
        return a[0] <= b[1] and b[0] <= a[1]

    def search_any(self, low, high):             # O(log n): one overlapping interval or None
        q, x = (low, high), self.root
        while x is not self.NIL and not self.overlaps(x.key, q):
            if x.left is not self.NIL and x.left.max >= low:
                x = x.left
            else:
                x = x.right
        return None if x is self.NIL else x.key

    def search_all(self, low, high):             # O(min(n, k log n)): all k overlaps, sorted by low
        out = []
        def visit(x):
            if x is self.NIL or x.max < low:     # nothing here reaches the query
                return
            visit(x.left)
            if x.key[0] > high:                  # this node and its right subtree start too late
                return
            if self.overlaps(x.key, (low, high)):
                out.append(x.key)
            visit(x.right)
        visit(self.root)
        return out

    def stab(self, point):                       # all intervals containing a point
        return self.search_all(point, point)

    def intervals(self):                         # traversal, sorted by low endpoint
        return list(self.inorder())

    # ------------------------------------------------ checks & drawing
    def validate(self):
        super().validate()
        def check(n):
            if n is self.NIL:
                return float("-inf")
            m = max(n.key[1], check(n.left), check(n.right))
            assert n.max == m, f"wrong max at {n.key}"
            return m
        check(self.root)
        return True

    def draw(self):
        draw_binary(self.root,
                    label=lambda n: f"[{n.key[0]},{n.key[1]}] max={n.max} ({'B' if n.color == BLACK else 'R'})",
                    is_empty=lambda n: n is self.NIL)

In [ ]:
it = IntervalTree()
banner("Insert: the classic CLRS example intervals")
for lo, hi in [(16, 21), (8, 9), (25, 30), (5, 8), (15, 23), (17, 19), (26, 26), (0, 3), (6, 10), (19, 20)]:
    it.add(lo, hi)
it.draw()
print("valid red-black tree with correct max fields?", it.validate())

banner("Search: any interval overlapping [22, 25]?  [11, 14]?")
print("search_any(22, 25) ->", it.search_any(22, 25))
print("search_any(11, 14) ->", it.search_any(11, 14))

banner("Search all overlaps / stabbing query")
print("search_all(8, 18) ->", it.search_all(8, 18))
print("stab(19)          ->", it.stab(19))

banner("Access / lookup: exact interval, earliest-starting interval")
print("contains [15,23]?", it.contains_interval(15, 23), "| contains [15,24]?", it.contains_interval(15, 24),
      "| earliest start:", it.min())

banner("Delete [15, 23] and [25, 30]; max fields are repaired automatically")
it.remove(15, 23); it.remove(25, 30)
it.draw()
print("valid?", it.validate(), "| search_all(8, 18) now ->", it.search_all(8, 18))

banner("Traversal: all intervals sorted by start")
print(it.intervals())


▶ Insert: the classic CLRS example intervals
[16,21] max=30 (B)
├── L: [8,9] max=23 (R)
│   ├── L: [5,8] max=10 (B)
│   │   ├── L: [0,3] max=3 (R)
│   │   └── R: [6,10] max=10 (R)
│   └── R: [15,23] max=23 (B)
└── R: [25,30] max=30 (R)
    ├── L: [17,19] max=20 (B)
    │   └── R: [19,20] max=20 (R)
    └── R: [26,26] max=26 (B)
valid red-black tree with correct max fields? True

▶ Search: any interval overlapping [22, 25]?  [11, 14]?
search_any(22, 25) -> (15, 23)
search_any(11, 14) -> None

▶ Search all overlaps / stabbing query
search_all(8, 18) -> [(5, 8), (6, 10), (8, 9), (15, 23), (16, 21), (17, 19)]
stab(19)          -> [(15, 23), (16, 21), (17, 19), (19, 20)]

▶ Access / lookup: exact interval, earliest-starting interval
contains [15,23]? True | contains [15,24]? False | earliest start: (0, 3)

▶ Delete [15, 23] and [25, 30]; max fields are repaired automatically
[16,21] max=26 (B)
├── L: [5,8] max=10 (R)
│   ├── L: [0,3] max=3 (B)
│   └── R: [8,9] max=10 (B)
│       └── L: [6,

### Application: meeting-room booking + randomized correctness check

In [ ]:
def book(calendar, start, end, title):
    clash = calendar.search_any(start, end)
    if clash:
        print(f"  ✗ {title:<14} {start:>4}-{end:<4} conflicts with {calendar.get(clash)} {clash}")
        return False
    calendar.add(start, end, title)
    print(f"  ✓ {title:<14} {start:>4}-{end:<4} booked")
    return True

room = IntervalTree()
for s, e, t in [(900, 959, "Standup"), (1000, 1129, "Design review"), (1100, 1159, "1:1"),
                (1130, 1229, "Lunch talk"), (1300, 1359, "Planning"), (1215, 1300, "Interview")]:
    book(room, s, e, t)

banner("Random test: 3000 mixed operations vs brute force")
it2, ref = IntervalTree(), set()
for step in range(3000):
    lo = random.randint(0, 500); hi = lo + random.randint(0, 60)
    r = random.random()
    if r < 0.5:
        it2.add(lo, hi); ref.add((lo, hi))
    elif r < 0.7 and ref:
        victim = random.choice(sorted(ref))
        it2.remove(*victim); ref.discard(victim)
    else:
        brute = sorted(iv for iv in ref if IntervalTree.overlaps(iv, (lo, hi)))
        assert it2.search_all(lo, hi) == brute
        hit = it2.search_any(lo, hi)
        assert (hit is None) == (not brute) and (hit is None or hit in brute)
    if step % 300 == 0:
        it2.validate()
it2.validate()
print(f"All queries matched brute force; final tree: {len(it2)} intervals, height {it2.height()}")

banner("Speed: overlap query vs scanning a list (20,000 intervals)")
ivs = [(lo, lo + random.randint(0, 20)) for lo in random.sample(range(1_000_000), 20_000)]
big_it = IntervalTree(ivs)
qs = [(x, x + 10) for x in random.sample(range(1_000_000), 500)]
t_tree = time_op("for a, b in Q: T.search_all(a, b)", T=big_it, Q=qs)
t_scan = time_op("for a, b in Q: [iv for iv in L if iv[0] <= b and a <= iv[1]]", L=ivs, Q=qs)
print(f"500 queries: interval tree {t_tree:.1f} ms vs linear scan {t_scan:.1f} ms")

  ✓ Standup         900-959  booked
  ✓ Design review  1000-1129 booked
  ✗ 1:1            1100-1159 conflicts with Design review (1000, 1129)
  ✓ Lunch talk     1130-1229 booked
  ✓ Planning       1300-1359 booked
  ✗ Interview      1215-1300 conflicts with Lunch talk (1130, 1229)

▶ Random test: 3000 mixed operations vs brute force


All queries matched brute force; final tree: 902 intervals, height 11

▶ Speed: overlap query vs scanning a list (20,000 intervals)


500 queries: interval tree 1.7 ms vs linear scan 382.5 ms


### Interval tree — complexity ($k$ = number of reported intervals)

| Operation | Time | Notes |
|---|---|---|
| insert interval | $O(\log n)$ | RB insert + $O(1)$ `max` repair per rotation + $O(\log n)$ path update |
| delete interval | $O(\log n)$ | RB delete, same repairs |
| exact lookup | $O(\log n)$ | ordinary BST search by `(low, high)` |
| find **any** overlap | $O(\log n)$ | one root-to-leaf walk guided by `max` |
| find **all** overlaps | $O(\min(n,\; k \log n))$ | pruning with `max` and `low` |
| stabbing query (point) | $O(\min(n,\; k \log n))$ | special case $[p, p]$ |
| min (earliest start) | $O(\log n)$ | |
| traversal | $O(n)$ | sorted by start |
| **Space** | $O(n)$ | one extra number (`max`) per node |

---
# Part 4 — Summary

## Master time-complexity cheat sheet

| Structure | Access / peek | Search | Insert | Delete | Specialized |
|---|---|---|---|---|---|
| **Stack** (array) | top $O(1)$ | $O(n)$ | push $O(1)$* | pop $O(1)$ | — |
| **Queue** (circular array) | front $O(1)$, index $O(1)$ | $O(n)$ | enqueue $O(1)$* | dequeue $O(1)$ | — |
| **Singly linked list** (head + tail) | index $O(n)$, ends $O(1)$ | $O(n)$ | prepend/append $O(1)$ | front $O(1)$, back $O(n)$ | reverse $O(n)$ |
| **Doubly linked list** | index $O(\min(i,n-i))$ | $O(n)$ | $O(1)$ at a known node | $O(1)$ at a known node | — |
| **General tree** | child/parent $O(1)$ | $O(n)$ | $O(1)$ | subtree $O(\deg)$ | traversals $O(n)$ |
| **Expression tree** | root $O(1)$ | $O(n)$ | substitute $O(n)$ | simplify $O(n)$ | build/convert/evaluate $O(n)$ |
| **Trie** | value $O(L)$ | $O(L)$ | $O(L)$ | $O(L)$ | prefix count $O(P)$ |
| **BST** | min/max $O(h)$ | $O(h)$ | $O(h)$ | $O(h)$ | $h$: $O(\log n)$ avg, $O(n)$ worst |
| **Binary heap** | min (or max) $O(1)$ | $O(n)$ | $O(\log n)$ | extract $O(\log n)$ | heapify $O(n)$, update-key $O(\log n)$ |
| **Disjoint sets** | find $O(\alpha(n))$† | connected $O(\alpha(n))$† | make_set $O(1)$ | lazy $O(\alpha(n))$† | union $O(\alpha(n))$† |
| **Red-black tree** | min/max $O(\log n)$ | $O(\log n)$ | $O(\log n)$ | $O(\log n)$ | rotation $O(1)$ |
| **Interval tree** | min $O(\log n)$ | overlap $O(\log n)$ | $O(\log n)$ | $O(\log n)$ | all overlaps $O(\min(n, k\log n))$ |

\* amortized  † amortized, with union by rank + path compression. All structures use $O(n)$ space (trie: $O(\text{total characters})$).

## Choosing a structure

| You need… | Use |
|---|---|
| undo / matching / DFS | **stack** |
| first-come-first-served / BFS | **queue** |
| many insertions/removals at positions you already hold | **(doubly) linked list** |
| hierarchy (files, org charts, parse trees) | **general tree** |
| parse / evaluate / transform formulas | **expression tree** |
| prefix search, autocomplete | **trie** |
| sorted data with guaranteed fast updates | **red-black tree** (a plain BST only for random data) |
| repeatedly take the smallest / largest | **heap** |
| dynamic connectivity, cycle detection, Kruskal | **disjoint sets** |
| overlap / scheduling conflicts | **interval tree** |

## Exercises

1. **Stack:** implement a `MinStack` whose `push`, `pop` and `get_min` are all $O(1)$. *(Hint: a second stack of running minimums.)*
2. **Queue:** implement a circular **deque** (insert/delete at both ends in $O(1)$) by extending `CircularQueue`.
3. **Linked list:** find the middle node in one pass; merge two sorted linked lists in $O(n+m)$.
4. **Structural induction:** prove that a binary tree of height $h$ has at most $2^h$ leaves, then verify it with `random_binary_tree`.
5. **Expression tree:** add support for **unary minus** (e.g. `-x + 3`) to the tokenizer, shunting-yard and evaluator.
6. **Trie:** add `delete_prefix(p)` that removes every word starting with `p` in $O(P)$ (careful with `prefix_count`!).
7. **BST:** write `is_bst` without an in-order traversal, using `(lo, hi)` bounds; prove it correct by structural induction.
8. **Heap:** implement a **d-ary heap** and measure how `push`/`pop` change for $d = 2, 4, 8$.
9. **Disjoint sets:** count the "islands" in a 2-D grid of 0/1 cells using union–find.
10. **Red-black tree:** add `rank(key)` and `select(k)` in $O(\log n)$ by augmenting each node with its subtree size (same hooks as the interval tree).
11. **Interval tree:** given a list of meetings, compute the minimum number of rooms needed. Compare an interval-tree solution with a heap-based one.